# Historical qwen collection and analysis

Preserved code order from the submitted working notebook. Outputs and transient metadata were removed. This is an execution history, NOT a fresh-runtime Run all entry point: later cells depend on earlier runtime state and restored environments. Use the numbered analysis notebooks for report analyses. GPU collection can be expensive; run selected collection sections deliberately.

### Original cell 0
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

OLD_DATA = Path(
    "/content/drive/MyDrive/B3IT_reproduction/data"
)

NEW_ROOT = Path(
    "/content/drive/MyDrive/B3IT_Qwen05_Noise_Pruning"
)
NEW_ROOT.mkdir(parents=True, exist_ok=True)

WORK = Path("/content/b3it_noise_pruning")
WORK.mkdir(parents=True, exist_ok=True)

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print("New results:", NEW_ROOT)
print("Existing data:", OLD_DATA)

### Original cell 1
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

### Original cell 2
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
REPO = Path("/content/b3it_noise_pruning_repo")
COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/timothee-chauvin/"
            "token-efficient-change-detection-llm-apis.git",
            str(REPO),
        ],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", COMMIT],
    check=True,
)

PROJECT = REPO / "track-llm-apis"
assert PROJECT.exists()

(NEW_ROOT / "repository_commit.txt").write_text(COMMIT)
print("Repository ready.")

### Original cell 3
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import shutil

subprocess.run(
    ["pip", "install", "-q", "uv"],
    check=True,
)

UV = shutil.which("uv")
assert UV

subprocess.run(
    [
        UV, "sync",
        "--project", str(PROJECT),
        "--python", "3.12",
        "--frozen",
        "--no-dev",
    ],
    check=True,
)

PYTHON = PROJECT / ".venv/bin/python"

subprocess.run(
    [
        UV, "pip", "install",
        "--python", str(PYTHON),
        "ninja",
    ],
    check=True,
)

print("Environment ready.")

### Original cell 4
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json

phase_dir = OLD_DATA / "bi_phase_1" / "phase_1b_T=0.0"
matches = []

for path in sorted(phase_dir.glob("*.json")):
    data = json.loads(path.read_text())
    if data.get("model_name") == MODEL:
        matches.append((path, data))

assert len(matches) == 1, (
    f"Expected one matching Phase 1b file, found {len(matches)}."
)

BI_PATH, bi_data = matches[0]

border_inputs = [
    prompt
    for prompt, counts in bi_data["results"].items()
    if len(counts) >= 2
]

assert len(border_inputs) >= 10

print("Source:", BI_PATH)
print("Available Border Inputs:", len(border_inputs))
print("First examples:", [repr(p) for p in border_inputs[:5]])

### Original cell 5
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
SMOKE_CONFIG = {
    "model": MODEL,
    "prompts": border_inputs[:10],
    "n_samples": 100,
    "temperature": 0.0,
    "batch_size": 64,
    "input_fraction": 0.95,
    "seed": 0,
    "datasets_dir": str(OLD_DATA / "datasets"),
}

TEST_JOBS = [
    {"name": "unchanged", "kind": "unchanged"},
    {
        "name": "noise_2pow_minus10",
        "kind": "random_noise",
        "scale": 2.0**-10,
    },
    {
        "name": "magnitude_2pow_minus10",
        "kind": "magnitude",
        "scale": 2.0**-10,
    },
    {
        "name": "random_pruning_2pow_minus10",
        "kind": "random",
        "scale": 2.0**-10,
    },
]

TEST_DIR = NEW_ROOT / "smoke_test_v1"
TEST_DIR.mkdir(parents=True, exist_ok=True)

config_path = TEST_DIR / "config.json"

if config_path.exists():
    assert json.loads(config_path.read_text()) == SMOKE_CONFIG, (
        "Existing test settings differ. Use a new test folder."
    )
else:
    config_path.write_text(
        json.dumps(SMOKE_CONFIG, indent=2)
    )

print("Test only: 10 prompts, 100 samples per model version.")

### Original cell 6
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
worker_code = r'''
import argparse
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch


def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=True))
    temporary.replace(path)


parser = argparse.ArgumentParser()
parser.add_argument("action", choices=["edit", "sample"])
parser.add_argument("--config", required=True)
parser.add_argument("--job", required=True)
parser.add_argument("--checkpoint", required=True)
parser.add_argument("--output", required=True)
args = parser.parse_args()

cfg = json.loads(Path(args.config).read_text())
job = json.loads(Path(args.job).read_text())

random.seed(cfg["seed"])
np.random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])

if args.action == "edit":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from torch.nn.utils import prune

    # Load only the model being modified.
    model = AutoModelForCausalLM.from_pretrained(
        cfg["model"],
        torch_dtype=torch.float16,
    ).to("cuda:0")
    tokenizer = AutoTokenizer.from_pretrained(cfg["model"])

    kind = job["kind"]
    scale = job.get("scale")

    with torch.no_grad():
        if kind == "random_noise":
            for _, parameter in model.named_parameters():
                if parameter.requires_grad:
                    parameter.add_(
                        torch.normal(
                            mean=0.0,
                            std=scale,
                            size=parameter.shape,
                            device=parameter.device,
                            dtype=parameter.dtype,
                        )
                    )

        elif kind in ("magnitude", "random"):
            method = (
                prune.l1_unstructured
                if kind == "magnitude"
                else prune.random_unstructured
            )

            for name, module in model.named_modules():
                if "mlp" not in name:
                    continue

                names = [
                    parameter_name
                    for parameter_name, _ in module.named_parameters(
                        recurse=False
                    )
                    if parameter_name == "weight"
                ]

                for parameter_name in names:
                    method(module, parameter_name, amount=scale)
                    prune.remove(module, parameter_name)

        else:
            raise ValueError(f"Unsupported perturbation: {kind}")

    model.eval()
    model.save_pretrained(args.checkpoint)
    tokenizer.save_pretrained(args.checkpoint)

    print("Perturbed checkpoint saved.", flush=True)

    # Process termination releases the editing model's GPU memory.

else:
    from vllm import LLM
    from track_llm_apis.sampling.vllm_sampling import (
        vllm_inference_bi,
    )
    from track_llm_apis.util import load_lmsys_chat_1m

    dataset = load_lmsys_chat_1m(
        use_cache=True,
        datasets_dir=Path(cfg["datasets_dir"]),
    )
    traffic = [
        row["conversation"][0]["content"]
        for row in dataset
    ]

    # This process loads only the inference model.
    model_path = (
        cfg["model"]
        if job["kind"] == "unchanged"
        else args.checkpoint
    )

    llm = LLM(
        model=model_path,
        dtype="float16",
        enforce_eager=True,
        gpu_memory_utilization=0.85,
    )

    result = vllm_inference_bi(
        llm=llm,
        inputs=cfg["prompts"],
        traffic_prompts=traffic,
        n_samples=cfg["n_samples"],
        temperature=cfg["temperature"],
        input_fraction=cfg["input_fraction"],
        batch_size=cfg["batch_size"],
        max_tokens=1,
    )

    payload = {
        "config": cfg,
        "job": job,
        "counts": result.counts,
        "prompt_lengths": result.prompt_lengths,
        "output_tokens": result.output_tokens,
    }

    write_json_atomic(args.output, payload)
    print("Recorded outputs saved:", args.output, flush=True)

    # Process termination releases vLLM and its GPU allocations.
'''

WORKER = WORK / "worker.py"
WORKER.write_text(worker_code)

import ast
ast.parse(worker_code)

# Keep a persistent copy of the exact worker.
(NEW_ROOT / "worker_v1.py").write_text(worker_code)

print("Worker created; syntax checked.")

### Original cell 7
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import os

ENV = os.environ.copy()
ENV.update({
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "TRACKLLM__DATA_DIR": str(OLD_DATA),
})

ENV["PATH"] = (
    str(PYTHON.parent)
    + os.pathsep
    + ENV.get("PATH", "")
)

def run_logged(arguments, log_path):
    command = [str(PYTHON), str(WORKER), *arguments]

    print("Starting:", arguments[0])
    print("Log:", log_path)

    with Path(log_path).open("w") as log:
        process = subprocess.Popen(
            command,
            cwd=PROJECT,
            env=ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()

            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

    if returncode != 0:
        raise RuntimeError(
            f"Process failed ({returncode}). See {log_path}"
        )

### Original cell 8
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import ast
import textwrap

source = WORKER.read_text()
marker = 'parser = argparse.ArgumentParser()'

if '\nif __name__ == "__main__":' in source:
    print("Main guard already present.")
else:
    assert marker in source, "Expected worker entry point not found."

    # Keep imports and helper functions at module level.
    # Move argument parsing and execution into main().
    prefix, body = source.split(marker, 1)

    corrected = (
        prefix
        + "\ndef main():\n"
        + textwrap.indent(marker + body, "    ")
        + '\n\nif __name__ == "__main__":\n'
        + "    main()\n"
    )

    ast.parse(corrected)

    backup = WORKER.with_name("worker_before_main_guard.py")
    if not backup.exists():
        backup.write_text(source)

    WORKER.write_text(corrected)
    (NEW_ROOT / "worker_v1.py").write_text(corrected)

    print("Main guard added; syntax checked.")

# Check that importing the worker does not start an experiment.
check = subprocess.run(
    [
        str(PYTHON), "-c",
        (
            "import runpy; "
            f"runpy.run_path({str(WORKER)!r}, "
            "run_name='worker_import_test'); "
            "print('Import check passed: no experiment launched.')"
        ),
    ],
    cwd=PROJECT,
    env=ENV,
    capture_output=True,
    text=True,
)

print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Worker import check failed.")

### Original cell 9
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
def valid_result(path, config, job):
    if not path.exists():
        return False

    try:
        saved = json.loads(path.read_text())

        if saved["config"] != config or saved["job"] != job:
            return False

        return all(
            sum(saved["counts"][prompt].values())
            == config["n_samples"]
            for prompt in config["prompts"]
        )
    except (KeyError, ValueError, TypeError):
        return False


for job in TEST_JOBS:
    output = TEST_DIR / f"{job['name']}.json"
    job_path = TEST_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    if valid_result(output, SMOKE_CONFIG, job):
        print("Already complete; skipping:", job["name"])
        continue

    checkpoint = WORK / "temporary_variant"

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    arguments = [
        "--config", str(config_path),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    print("\nMODEL VERSION:", job["name"])

    if job["kind"] != "unchanged":
        run_logged(
            ["edit", *arguments],
            TEST_DIR / f"{job['name']}_edit.log",
        )

    # The editing process has exited before inference starts.
    run_logged(
        ["sample", *arguments],
        TEST_DIR / f"{job['name']}_sample.log",
    )

    assert valid_result(output, SMOKE_CONFIG, job)

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    print("Completed and verified:", job["name"])

print("\nAll four smoke-test collections completed.")

### Original cell 10
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
for job in TEST_JOBS:
    path = TEST_DIR / f"{job['name']}.json"
    assert valid_result(path, SMOKE_CONFIG, job), (
        f"Incomplete result: {job['name']}"
    )

    saved = json.loads(path.read_text())
    counts = saved["counts"]

    print(
        job["name"],
        "| prompts:", len(counts),
        "| samples per prompt:",
        sorted({sum(c.values()) for c in counts.values()}),
        "| prompts with multiple observed outputs:",
        sum(len(c) > 1 for c in counts.values()),
    )

print("\nSaved in:", TEST_DIR)

### Original cell 11
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

collections = {}

for job in TEST_JOBS:
    path = TEST_DIR / f"{job['name']}.json"
    assert valid_result(path, SMOKE_CONFIG, job)
    collections[job["name"]] = json.loads(path.read_text())

prompts = SMOKE_CONFIG["prompts"]
names = [job["name"] for job in TEST_JOBS]

# Give each observed output a shared integer ID for each prompt.
vocabularies = {}
for prompt in prompts:
    outputs = {
        token
        for name in names
        for token in collections[name]["counts"][prompt]
    }
    vocabularies[prompt] = {
        token: i for i, token in enumerate(sorted(outputs))
    }

# Reconstruct sample pools from counts.
# Only frequencies are needed for the TV statistic.
pools = {}
for name in names:
    pools[name] = {}
    for prompt in prompts:
        vocab = vocabularies[prompt]
        counts = collections[name]["counts"][prompt]

        pools[name][prompt] = np.concatenate([
            np.full(count, vocab[token], dtype=np.int32)
            for token, count in counts.items()
        ])

print("Loaded four collections. No model inference required.")

### Original cell 12
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
rng = np.random.default_rng(0)

N_TESTS = 1000
N_REFERENCE = 50
PROMPT_COUNTS = [1, 3, 5, 10]
DETECTION_COUNTS = [1, 3, 10, 20, 50]

def tv_distance(a, b, vocab_size):
    pa = np.bincount(a, minlength=vocab_size) / len(a)
    pb = np.bincount(b, minlength=vocab_size) / len(b)
    return 0.5 * np.abs(pa - pb).sum()

def score(selected, left, right, n_detection, same):
    distances = []

    for prompt in selected:
        a_pool = pools[left][prompt]
        b_pool = pools[right][prompt]

        if same:
            indices = rng.choice(
                len(a_pool),
                size=N_REFERENCE + n_detection,
                replace=False,
            )
            a = a_pool[indices[:N_REFERENCE]]
            b = a_pool[indices[N_REFERENCE:]]
        else:
            a = rng.choice(a_pool, N_REFERENCE, replace=False)
            b = rng.choice(b_pool, n_detection, replace=False)

        distances.append(
            tv_distance(a, b, len(vocabularies[prompt]))
        )

    return float(np.mean(distances))

records = []

for n_prompts in PROMPT_COUNTS:
    for n_detection in DETECTION_COUNTS:
        for variant in names:
            if variant == "unchanged":
                continue

            h0_original = []
            h0_variant = []
            h1 = []

            for _ in range(N_TESTS):
                selected = rng.choice(
                    prompts, size=n_prompts, replace=False
                )

                h0_original.append(score(
                    selected, "unchanged", "unchanged",
                    n_detection, same=True,
                ))
                h0_variant.append(score(
                    selected, variant, variant,
                    n_detection, same=True,
                ))
                h1.append(score(
                    selected, "unchanged", variant,
                    n_detection, same=False,
                ))

            h0 = h0_original + h0_variant
            auc = roc_auc_score(
                [0] * len(h0) + [1] * len(h1),
                h0 + h1,
            )

            records.append({
                "variant": variant,
                "n_prompts": n_prompts,
                "detection_samples_per_prompt": n_detection,
                "auc": auc,
            })

        print(
            "Completed:",
            n_prompts, "prompts ×",
            n_detection, "detection samples",
        )

per_variant = pd.DataFrame(records)
per_variant.to_csv(
    TEST_DIR / "smoke_auc_per_variant.csv",
    index=False,
)

display(per_variant.head())

### Original cell 13
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

summary = (
    per_variant
    .groupby(
        ["n_prompts", "detection_samples_per_prompt"],
        as_index=False,
    )["auc"]
    .mean()
)

baseline_lengths = collections["unchanged"]["prompt_lengths"]
mean_input_tokens = np.mean([
    baseline_lengths[p] for p in prompts
])

summary["yearly_cost"] = (
    8760
    * summary["n_prompts"]
    * summary["detection_samples_per_prompt"]
    * (3e-6 * mean_input_tokens + 12e-6)
)

colors = {
    1: "#E8C75A",
    3: "#FFB43B",
    5: "#F56600",
    10: "#963300",
}
sizes = dict(zip(DETECTION_COUNTS, [15, 25, 35, 48, 65]))

fig, ax = plt.subplots(figsize=(9, 5))

for n_prompts, group in summary.groupby("n_prompts"):
    group = group.sort_values("yearly_cost")

    ax.plot(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        label=f"{n_prompts} prompts",
    )
    ax.scatter(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        s=[
            sizes[n]
            for n in group["detection_samples_per_prompt"]
        ],
    )

ax.set_xscale("log")
ax.set_xlabel("Estimated annual monitoring cost ($)")
ax.set_ylabel("Mean ROC AUC over three test variants")
ax.set_title(
    "Smoke test — Qwen2.5-0.5B / T4 / FP16\n"
    "One noise variant and two pruning variants"
)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()

summary.to_csv(TEST_DIR / "smoke_auc_summary.csv", index=False)
fig.savefig(TEST_DIR / "smoke_result.png", dpi=300)
fig.savefig(TEST_DIR / "smoke_result.pdf")

plt.show()
display(summary)

### Original cell 14
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
comparison = per_variant[
    (per_variant["n_prompts"] == 5)
    & (per_variant["detection_samples_per_prompt"] == 3)
][["variant", "auc"]]

display(comparison)

### Original cell 15
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import shutil
import hashlib
from pathlib import Path

EXPANDED_DIR = NEW_ROOT / "expanded_scales_v1"
EXPANDED_DIR.mkdir(parents=True, exist_ok=True)

# Preserve exactly the smoke-test collection settings.
EXPANDED_CONFIG = json.loads(
    (TEST_DIR / "config.json").read_text()
)

EXPANDED_CONFIG_PATH = EXPANDED_DIR / "config.json"

if EXPANDED_CONFIG_PATH.exists():
    assert (
        json.loads(EXPANDED_CONFIG_PATH.read_text())
        == EXPANDED_CONFIG
    ), "Existing experiment has different settings."
else:
    EXPANDED_CONFIG_PATH.write_text(
        json.dumps(EXPANDED_CONFIG, indent=2)
    )

EXPANDED_JOBS = [{"name": "unchanged", "kind": "unchanged"}]

for exponent in [0, 5, 10, 15, 20, 25, 30]:
    EXPANDED_JOBS.append({
        "name": f"noise_2pow_minus{exponent}",
        "kind": "random_noise",
        "scale": 2.0 ** -exponent,
    })

for kind, prefix in [
    ("magnitude", "magnitude"),
    ("random", "random_pruning"),
]:
    for exponent in [0, 5, 10, 15, 20]:
        EXPANDED_JOBS.append({
            "name": f"{prefix}_2pow_minus{exponent}",
            "kind": kind,
            "scale": 2.0 ** -exponent,
        })

manifest = {
    "config": EXPANDED_CONFIG,
    "jobs": EXPANDED_JOBS,
    "worker_sha256": hashlib.sha256(
        WORKER.read_bytes()
    ).hexdigest(),
}

manifest_path = EXPANDED_DIR / "manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        "The worker or experiment settings changed. "
        "Use a new experiment folder."
    )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

# Preserve the corrected worker, including its main guard.
EXPANDED_WORKER = EXPANDED_DIR / "worker.py"
if not EXPANDED_WORKER.exists():
    shutil.copy2(WORKER, EXPANDED_WORKER)

print("Perturbed variants:", len(EXPANDED_JOBS) - 1)
print("Total versions including baseline:", len(EXPANDED_JOBS))
print("Prompts:", len(EXPANDED_CONFIG["prompts"]))
print("Samples per prompt:", EXPANDED_CONFIG["n_samples"])

### Original cell 16
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
reused = 0

for job in EXPANDED_JOBS:
    source = TEST_DIR / f"{job['name']}.json"
    destination = EXPANDED_DIR / f"{job['name']}.json"

    if valid_result(destination, EXPANDED_CONFIG, job):
        print("Already available:", job["name"])
        continue

    if valid_result(source, EXPANDED_CONFIG, job):
        shutil.copy2(source, destination)
        reused += 1
        print("Reused:", job["name"])

print("Collections copied:", reused)

### Original cell 17
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import time

# run_logged() uses the global WORKER variable.
WORKER = EXPANDED_WORKER

started = time.monotonic()

for position, job in enumerate(EXPANDED_JOBS, start=1):
    output = EXPANDED_DIR / f"{job['name']}.json"

    if valid_result(output, EXPANDED_CONFIG, job):
        print(f"[{position}/{len(EXPANDED_JOBS)}] SKIP:", job["name"])
        continue

    job_path = EXPANDED_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    checkpoint = WORK / "temporary_expanded_variant"
    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    arguments = [
        "--config", str(EXPANDED_CONFIG_PATH),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    print(
        f"\n[{position}/{len(EXPANDED_JOBS)}]",
        job["name"],
        "| scale:", job.get("scale"),
    )
    variant_started = time.monotonic()

    if job["kind"] != "unchanged":
        run_logged(
            ["edit", *arguments],
            EXPANDED_DIR / f"{job['name']}_edit.log",
        )

    run_logged(
        ["sample", *arguments],
        EXPANDED_DIR / f"{job['name']}_sample.log",
    )

    assert valid_result(output, EXPANDED_CONFIG, job), (
        f"Invalid collection: {job['name']}"
    )

    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    print(
        "Saved and verified.",
        "Minutes:",
        round((time.monotonic() - variant_started) / 60, 1),
    )

print(
    "\nCollection finished. Elapsed minutes:",
    round((time.monotonic() - started) / 60, 1),
)

### Original cell 18
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
verification = []

for job in EXPANDED_JOBS:
    path = EXPANDED_DIR / f"{job['name']}.json"
    complete = valid_result(path, EXPANDED_CONFIG, job)

    row = {
        "variant": job["name"],
        "kind": job["kind"],
        "scale": job.get("scale"),
        "complete": complete,
    }

    if complete:
        saved = json.loads(path.read_text())
        row["variable_output_prompts"] = sum(
            len(counts) > 1
            for counts in saved["counts"].values()
        )

    verification.append(row)

verification_df = pd.DataFrame(verification)
display(verification_df)

assert verification_df["complete"].all(), (
    "Some collections are incomplete. Rerun Cell 16."
)

verification_df.to_csv(
    EXPANDED_DIR / "collection_verification.csv",
    index=False,
)

print("All 18 model versions are saved.")

### Original cell 19
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASELINE = json.loads(
    (EXPANDED_DIR / "unchanged.json").read_text()
)

N_TESTS = 1000
N_REFERENCE = 50
N_PROMPTS = 5
N_DETECTION = 3

ANALYSIS_DIR = EXPANDED_DIR / "analysis_5prompts_3samples_v1"
ANALYSIS_DIR.mkdir(exist_ok=True)

def evaluate_variant(baseline, variant, seed):
    rng = np.random.default_rng(seed)
    prompts = EXPANDED_CONFIG["prompts"]

    baseline_pools = {}
    variant_pools = {}
    vocabulary_sizes = {}

    for prompt in prompts:
        a_counts = baseline["counts"][prompt]
        b_counts = variant["counts"][prompt]

        vocabulary = {
            token: index
            for index, token in enumerate(
                sorted(set(a_counts) | set(b_counts))
            )
        }
        vocabulary_sizes[prompt] = len(vocabulary)

        def expand(counts):
            return np.concatenate([
                np.full(count, vocabulary[token], dtype=np.int32)
                for token, count in counts.items()
            ])

        baseline_pools[prompt] = expand(a_counts)
        variant_pools[prompt] = expand(b_counts)

    def statistic(selected, left, right, same):
        distances = []

        for prompt in selected:
            a_pool = left[prompt]
            b_pool = right[prompt]

            if same:
                indices = rng.choice(
                    len(a_pool),
                    N_REFERENCE + N_DETECTION,
                    replace=False,
                )
                a = a_pool[indices[:N_REFERENCE]]
                b = a_pool[indices[N_REFERENCE:]]
            else:
                a = rng.choice(a_pool, N_REFERENCE, replace=False)
                b = rng.choice(b_pool, N_DETECTION, replace=False)

            size = vocabulary_sizes[prompt]
            pa = np.bincount(a, minlength=size) / len(a)
            pb = np.bincount(b, minlength=size) / len(b)

            distances.append(0.5 * np.abs(pa - pb).sum())

        return float(np.mean(distances))

    h0 = []
    h1 = []

    for _ in range(N_TESTS):
        selected = rng.choice(
            prompts, N_PROMPTS, replace=False
        )

        h0.append(statistic(
            selected, baseline_pools, baseline_pools, True
        ))
        h0.append(statistic(
            selected, variant_pools, variant_pools, True
        ))
        h1.append(statistic(
            selected, baseline_pools, variant_pools, False
        ))

    auc = roc_auc_score(
        [0] * len(h0) + [1] * len(h1),
        h0 + h1,
    )

    return float(auc)


analysis_rows = []

for job in EXPANDED_JOBS:
    if job["kind"] == "unchanged":
        continue

    result_path = ANALYSIS_DIR / f"{job['name']}.json"

    if result_path.exists():
        row = json.loads(result_path.read_text())
    else:
        variant = json.loads(
            (EXPANDED_DIR / f"{job['name']}.json").read_text()
        )

        seed = int(
            hashlib.sha256(job["name"].encode()).hexdigest()[:8],
            16,
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": N_PROMPTS,
            "detection_samples_per_prompt": N_DETECTION,
            "reference_samples_per_prompt": N_REFERENCE,
            "n_tests": N_TESTS,
            "auc": evaluate_variant(BASELINE, variant, seed),
        }

        result_path.write_text(json.dumps(row, indent=2))

    analysis_rows.append(row)
    print(row["variant"], "| AUC:", round(row["auc"], 4))

difficulty_results = pd.DataFrame(analysis_rows)
difficulty_results.to_csv(
    ANALYSIS_DIR / "auc_by_strength.csv",
    index=False,
)

display(difficulty_results[["kind", "scale", "auc"]])

### Original cell 20
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

panels = [
    ("random_noise", "Gaussian noise", "Noise standard deviation"),
    ("magnitude", "Magnitude pruning", "Fraction of MLP weights pruned"),
    ("random", "Random pruning", "Fraction of MLP weights pruned"),
]

fig_strength, axes = plt.subplots(
    1, 3, figsize=(15, 4.5), sharey=True
)

for ax, (kind, title, xlabel) in zip(axes, panels):
    subset = (
        difficulty_results[difficulty_results["kind"] == kind]
        .sort_values("scale")
    )

    ax.plot(
        subset["scale"],
        subset["auc"],
        marker="o",
        linewidth=1.5,
    )
    ax.axhline(
        0.5, color="gray", linestyle="--",
        linewidth=1, label="Random guessing",
    )
    ax.set_xscale("log")
    ax.set_ylim(0.45, 1.02)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(alpha=0.25)

axes[0].set_ylabel("ROC AUC")
axes[-1].legend()

fig_strength.suptitle(
    "Qwen2.5-0.5B — expanded noise/pruning experiment\n"
    "5 prompts × 3 detection samples; 50 reference samples; T4 / FP16"
)

fig_strength.tight_layout()

fig_strength.savefig(
    ANALYSIS_DIR / "auc_by_strength.png",
    dpi=300,
    bbox_inches="tight",
)
fig_strength.savefig(
    ANALYSIS_DIR / "auc_by_strength.pdf",
    bbox_inches="tight",
)

plt.show()

### Original cell 21
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

SWEEP_DIR = EXPANDED_DIR / "figure1_sweep_v1"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

SWEEP_SETTINGS = {
    "n_tests": 1000,
    "reference_samples": 50,
    "prompt_counts": [1, 3, 5, 10],
    "detection_counts": [1, 3, 10, 20, 50],
    "seed": 0,
    "h0": "pooled unchanged-vs-itself and variant-vs-itself",
    "aggregation": "equal mean over 17 variants",
}

saved_collections = {}

for job in EXPANDED_JOBS:
    path = EXPANDED_DIR / f"{job['name']}.json"
    assert valid_result(path, EXPANDED_CONFIG, job), (
        f"Missing or incomplete collection: {job['name']}"
    )
    saved_collections[job["name"]] = json.loads(path.read_text())

# Record input checksums so cached analyses cannot silently use changed data.
manifest = {
    "settings": SWEEP_SETTINGS,
    "input_sha256": {
        job["name"]: hashlib.sha256(
            (EXPANDED_DIR / f"{job['name']}.json").read_bytes()
        ).hexdigest()
        for job in EXPANDED_JOBS
    },
}

manifest_path = SWEEP_DIR / "analysis_manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, (
        "Inputs or analysis settings changed. Use a new sweep folder."
    )
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))

print("Verified collections:", len(saved_collections))
print("Configurations per variant: 20")
print("Total variant/configuration evaluations: 340")

### Original cell 22
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
def compute_configuration_auc(
    baseline,
    variant,
    n_prompts,
    n_detection,
    n_tests,
    seed,
):
    rng = np.random.default_rng(seed)
    prompts = EXPANDED_CONFIG["prompts"]
    n_reference = SWEEP_SETTINGS["reference_samples"]

    base_pools = {}
    variant_pools = {}
    vocab_sizes = {}

    for prompt in prompts:
        a_counts = baseline["counts"][prompt]
        b_counts = variant["counts"][prompt]

        vocab = {
            token: index
            for index, token in enumerate(
                sorted(set(a_counts) | set(b_counts))
            )
        }
        vocab_sizes[prompt] = len(vocab)

        def expand(counts):
            return np.concatenate([
                np.full(count, vocab[token], dtype=np.int32)
                for token, count in counts.items()
            ])

        base_pools[prompt] = expand(a_counts)
        variant_pools[prompt] = expand(b_counts)

        assert len(base_pools[prompt]) >= n_reference + n_detection
        assert len(variant_pools[prompt]) >= n_reference + n_detection

    def score(selected, left, right, same):
        distances = []

        for prompt in selected:
            a_pool = left[prompt]
            b_pool = right[prompt]

            if same:
                indices = rng.choice(
                    len(a_pool),
                    n_reference + n_detection,
                    replace=False,
                )
                a = a_pool[indices[:n_reference]]
                b = a_pool[indices[n_reference:]]
            else:
                a = rng.choice(
                    a_pool, n_reference, replace=False
                )
                b = rng.choice(
                    b_pool, n_detection, replace=False
                )

            size = vocab_sizes[prompt]
            pa = np.bincount(a, minlength=size) / len(a)
            pb = np.bincount(b, minlength=size) / len(b)

            distances.append(0.5 * np.abs(pa - pb).sum())

        return float(np.mean(distances))

    h0 = []
    h1 = []

    for _ in range(n_tests):
        selected = rng.choice(
            prompts, n_prompts, replace=False
        )

        h0.append(score(
            selected, base_pools, base_pools, True
        ))
        h0.append(score(
            selected, variant_pools, variant_pools, True
        ))
        h1.append(score(
            selected, base_pools, variant_pools, False
        ))

    return float(roc_auc_score(
        [0] * len(h0) + [1] * len(h1),
        h0 + h1,
    ))

print("Evaluation function ready.")

### Original cell 23
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from tqdm.auto import tqdm

baseline = saved_collections["unchanged"]

tasks = [
    (job, n_prompts, n_detection)
    for job in EXPANDED_JOBS
    if job["kind"] != "unchanged"
    for n_prompts in SWEEP_SETTINGS["prompt_counts"]
    for n_detection in SWEEP_SETTINGS["detection_counts"]
]

rows = []

for job, n_prompts, n_detection in tqdm(
    tasks, desc="B3IT evaluations"
):
    filename = (
        f"{job['name']}"
        f"_p{n_prompts}_d{n_detection}.json"
    )
    result_path = SWEEP_DIR / filename

    if result_path.exists():
        row = json.loads(result_path.read_text())
    else:
        key = (
            f"{SWEEP_SETTINGS['seed']}:"
            f"{job['name']}:{n_prompts}:{n_detection}"
        )
        seed = int(
            hashlib.sha256(key.encode()).hexdigest()[:8], 16
        )

        auc = compute_configuration_auc(
            baseline=baseline,
            variant=saved_collections[job["name"]],
            n_prompts=n_prompts,
            n_detection=n_detection,
            n_tests=SWEEP_SETTINGS["n_tests"],
            seed=seed,
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": n_prompts,
            "detection_samples_per_prompt": n_detection,
            "auc": auc,
        }

        temporary = result_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(row, indent=2))
        temporary.replace(result_path)

    rows.append(row)

sweep_results = pd.DataFrame(rows)
assert len(sweep_results) == 340

sweep_results.to_csv(
    SWEEP_DIR / "auc_per_variant_and_configuration.csv",
    index=False,
)

print("All 340 evaluations completed and saved.")

### Original cell 24
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
summary = (
    sweep_results
    .groupby(
        ["n_prompts", "detection_samples_per_prompt"],
        as_index=False,
    )
    .agg(
        auc=("auc", "mean"),
        n_variants=("variant", "nunique"),
    )
)

assert len(summary) == 20
assert (summary["n_variants"] == 17).all()

mean_input_tokens = np.mean([
    baseline["prompt_lengths"][prompt]
    for prompt in EXPANDED_CONFIG["prompts"]
])

summary["yearly_cost"] = (
    8760
    * summary["n_prompts"]
    * summary["detection_samples_per_prompt"]
    * (3e-6 * mean_input_tokens + 12e-6)
)

summary.to_csv(
    SWEEP_DIR / "figure1_partial_summary.csv",
    index=False,
)

display(summary)

### Original cell 25
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

colors = {
    1: "#E8C75A",
    3: "#FFB43B",
    5: "#F56600",
    10: "#963300",
}
sizes = dict(zip(
    [1, 3, 10, 20, 50],
    [15, 25, 35, 48, 65],
))

fig_cost, ax = plt.subplots(figsize=(10, 6))
fig_cost.subplots_adjust(right=0.75, bottom=0.22)

for n_prompts, group in summary.groupby("n_prompts"):
    group = group.sort_values("yearly_cost")

    ax.plot(
        group["yearly_cost"],
        group["auc"],
        color=colors[n_prompts],
        linewidth=1.3,
    )
    ax.scatter(
        group["yearly_cost"],
        group["auc"],
        color=colors[n_prompts],
        s=[
            sizes[n]
            for n in group["detection_samples_per_prompt"]
        ],
        zorder=3,
    )

ax.set_xscale("log")
ax.set_xlabel("Estimated annual monitoring cost ($)")
ax.set_ylabel("Mean ROC AUC over 17 variants")
ax.set_title(
    "Partial Figure 1 — Qwen2.5-0.5B\n"
    "Gaussian noise, magnitude pruning and random pruning"
)
ax.grid(alpha=0.25)
ax.set_axisbelow(True)

prompt_legend = ax.legend(
    handles=[
        Line2D(
            [0], [0],
            color=colors[p],
            label=str(p),
        )
        for p in colors
    ],
    title="Number of prompts",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)
ax.add_artist(prompt_legend)

ax.legend(
    handles=[
        Line2D(
            [0], [0],
            linestyle="",
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=np.sqrt(sizes[n]),
            label=str(n),
        )
        for n in sizes
    ],
    title="Detection samples\nper prompt",
    bbox_to_anchor=(1.02, 0.55),
    loc="upper left",
    frameon=False,
)

fig_cost.text(
    0.12, 0.05,
    "T4 / FP16; 10 Border Inputs; 17 selected variants.\n"
    "Independent empirical bootstrap analysis; "
    "100 recorded outputs per prompt.\n"
    "50 reference samples per test; equal averaging over variants.",
    fontsize=9,
)

fig_cost.savefig(
    SWEEP_DIR / "figure1_partial_noise_pruning.png",
    dpi=300,
    bbox_inches="tight",
)
fig_cost.savefig(
    SWEEP_DIR / "figure1_partial_noise_pruning.pdf",
    bbox_inches="tight",
)

plt.show()
print("Saved results and figures in:", SWEEP_DIR)

### Original cell 26
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import pandas as pd

comparison = (
    sweep_results[
        (sweep_results["n_prompts"] == 10)
        & sweep_results["detection_samples_per_prompt"].isin([20, 50])
    ]
    .pivot(
        index=["variant", "kind", "scale"],
        columns="detection_samples_per_prompt",
        values="auc",
    )
    .rename(columns={
        20: "auc_20_samples",
        50: "auc_50_samples",
    })
    .reset_index()
)

comparison["difference_50_minus_20"] = (
    comparison["auc_50_samples"]
    - comparison["auc_20_samples"]
)

comparison = comparison.sort_values("difference_50_minus_20")

display(comparison)

comparison.to_csv(
    SWEEP_DIR / "comparison_20_vs_50_samples.csv",
    index=False,
)

### Original cell 27
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
family_comparison = (
    comparison.groupby("kind", as_index=False)[
        [
            "auc_20_samples",
            "auc_50_samples",
            "difference_50_minus_20",
        ]
    ]
    .mean()
)

display(family_comparison)

### Original cell 28
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
control_rows = []

for detection_samples in [3, 20, 50]:
    control_auc = compute_configuration_auc(
        baseline=baseline,
        variant=baseline,
        n_prompts=10,
        n_detection=detection_samples,
        n_tests=1000,
        seed=12345 + detection_samples,
    )

    control_rows.append({
        "n_prompts": 10,
        "detection_samples_per_prompt": detection_samples,
        "unchanged_vs_unchanged_auc": control_auc,
    })

control_results = pd.DataFrame(control_rows)
display(control_results)

control_results.to_csv(
    SWEEP_DIR / "unchanged_control.csv",
    index=False,
)

### Original cell 29
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
def compute_configuration_auc(
    baseline,
    variant,
    n_prompts,
    n_detection,
    n_tests,
    seed,
):
    rng = np.random.default_rng(seed)
    prompts = EXPANDED_CONFIG["prompts"]
    n_reference = SWEEP_SETTINGS["reference_samples"]

    baseline_probabilities = {}
    variant_probabilities = {}

    for prompt in prompts:
        a_counts = baseline["counts"][prompt]
        b_counts = variant["counts"][prompt]
        tokens = sorted(set(a_counts) | set(b_counts))

        a = np.array(
            [a_counts.get(token, 0) for token in tokens],
            dtype=float,
        )
        b = np.array(
            [b_counts.get(token, 0) for token in tokens],
            dtype=float,
        )

        baseline_probabilities[prompt] = a / a.sum()
        variant_probabilities[prompt] = b / b.sum()

    def score(selected, left, right):
        distances = []

        for prompt in selected:
            # Independent multinomial draws are equivalent to
            # sampling with replacement from the recorded outputs.
            reference = rng.multinomial(
                n_reference, left[prompt]
            ) / n_reference

            detection = rng.multinomial(
                n_detection, right[prompt]
            ) / n_detection

            distances.append(
                0.5 * np.abs(reference - detection).sum()
            )

        return float(np.mean(distances))

    h0 = []
    h1 = []

    for _ in range(n_tests):
        selected = rng.choice(
            prompts, n_prompts, replace=False
        )

        h0.append(score(
            selected,
            baseline_probabilities,
            baseline_probabilities,
        ))
        h0.append(score(
            selected,
            variant_probabilities,
            variant_probabilities,
        ))
        h1.append(score(
            selected,
            baseline_probabilities,
            variant_probabilities,
        ))

    return float(roc_auc_score(
        [0] * len(h0) + [1] * len(h1),
        h0 + h1,
    ))

print("Corrected independent empirical sampling is ready.")

### Original cell 30
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
corrected_controls = []

for n_detection in [3, 20, 50]:
    auc = compute_configuration_auc(
        baseline=baseline,
        variant=baseline,
        n_prompts=10,
        n_detection=n_detection,
        n_tests=3000,
        seed=12345 + n_detection,
    )

    corrected_controls.append({
        "detection_samples_per_prompt": n_detection,
        "unchanged_vs_unchanged_auc": auc,
    })

corrected_control_df = pd.DataFrame(corrected_controls)
display(corrected_control_df)

assert corrected_control_df[
    "unchanged_vs_unchanged_auc"
].between(0.45, 0.55).all(), (
    "Control is still far from 0.5. "
    "Stop here and share the output."
)

### Original cell 31
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
SWEEP_DIR = EXPANDED_DIR / "figure1_sweep_v2_independent"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

corrected_manifest = {
    **manifest,
    "sampling_method": (
        "Independent multinomial draws from empirical "
        "distributions for both H0 and H1"
    ),
}

manifest_path = SWEEP_DIR / "analysis_manifest.json"

if manifest_path.exists():
    assert (
        json.loads(manifest_path.read_text())
        == corrected_manifest
    ), "Analysis settings differ from existing results."
else:
    manifest_path.write_text(
        json.dumps(corrected_manifest, indent=2)
    )

corrected_control_df.to_csv(
    SWEEP_DIR / "unchanged_control.csv",
    index=False,
)

tasks = [
    (job, n_prompts, n_detection)
    for job in EXPANDED_JOBS
    if job["kind"] != "unchanged"
    for n_prompts in SWEEP_SETTINGS["prompt_counts"]
    for n_detection in SWEEP_SETTINGS["detection_counts"]
]

rows = []

for job, n_prompts, n_detection in tqdm(
    tasks, desc="Corrected B3IT evaluations"
):
    path = SWEEP_DIR / (
        f"{job['name']}_p{n_prompts}_d{n_detection}.json"
    )

    if path.exists():
        row = json.loads(path.read_text())
    else:
        key = (
            f"v2:{SWEEP_SETTINGS['seed']}:"
            f"{job['name']}:{n_prompts}:{n_detection}"
        )
        seed = int(
            hashlib.sha256(key.encode()).hexdigest()[:8], 16
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": n_prompts,
            "detection_samples_per_prompt": n_detection,
            "auc": compute_configuration_auc(
                baseline=baseline,
                variant=saved_collections[job["name"]],
                n_prompts=n_prompts,
                n_detection=n_detection,
                n_tests=SWEEP_SETTINGS["n_tests"],
                seed=seed,
            ),
        }

        temporary = path.with_suffix(".tmp")
        temporary.write_text(json.dumps(row, indent=2))
        temporary.replace(path)

    rows.append(row)

sweep_results = pd.DataFrame(rows)
assert len(sweep_results) == 340

sweep_results.to_csv(
    SWEEP_DIR / "auc_per_variant_and_configuration.csv",
    index=False,
)

print("Corrected analysis saved:", SWEEP_DIR)

### Original cell 32
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import hashlib

BASELINE_CHECK_DIR = NEW_ROOT / "independent_baseline_check_v1"
BASELINE_CHECK_DIR.mkdir(parents=True, exist_ok=True)

original_baseline_path = EXPANDED_DIR / "unchanged.json"
original_baseline = json.loads(
    original_baseline_path.read_text()
)

second_config = dict(original_baseline["config"])
second_config["seed"] = original_baseline["config"]["seed"] + 1

second_job = {
    "name": "unchanged_independent",
    "kind": "unchanged",
}

second_config_path = BASELINE_CHECK_DIR / "config.json"
second_job_path = BASELINE_CHECK_DIR / "job.json"
second_output_path = BASELINE_CHECK_DIR / "unchanged_independent.json"

for path, content in [
    (second_config_path, second_config),
    (second_job_path, second_job),
]:
    if path.exists():
        assert json.loads(path.read_text()) == content, (
            f"Existing settings differ: {path}"
        )
    else:
        path.write_text(json.dumps(content, indent=2))

# Use the exact worker saved with the expanded experiment.
WORKER = EXPANDED_DIR / "worker.py"
assert WORKER.exists()

experiment_manifest = json.loads(
    (EXPANDED_DIR / "manifest.json").read_text()
)
assert hashlib.sha256(
    WORKER.read_bytes()
).hexdigest() == experiment_manifest["worker_sha256"]

print("Original seed:", original_baseline["config"]["seed"])
print("Second seed:", second_config["seed"])
print("Prompts:", len(second_config["prompts"]))
print("Samples per prompt:", second_config["n_samples"])

### Original cell 33
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
if valid_result(
    second_output_path, second_config, second_job
):
    print("Second baseline already collected; skipping.")
else:
    run_logged(
        [
            "sample",
            "--config", str(second_config_path),
            "--job", str(second_job_path),
            "--checkpoint", str(
                WORK / "unused_baseline_checkpoint"
            ),
            "--output", str(second_output_path),
        ],
        BASELINE_CHECK_DIR / "collection.log",
    )

assert valid_result(
    second_output_path, second_config, second_job
)

print("Second baseline saved and verified.")

### Original cell 34
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import numpy as np
import pandas as pd

second_baseline = json.loads(
    second_output_path.read_text()
)

baseline_pairs = []
comparison_rows = []

for index, prompt in enumerate(second_config["prompts"]):
    first_counts = original_baseline["counts"][prompt]
    second_counts = second_baseline["counts"][prompt]

    tokens = sorted(set(first_counts) | set(second_counts))

    first = np.array(
        [first_counts.get(token, 0) for token in tokens],
        dtype=np.int64,
    )
    second = np.array(
        [second_counts.get(token, 0) for token in tokens],
        dtype=np.int64,
    )

    tv = 0.5 * np.abs(
        first / first.sum() - second / second.sum()
    ).sum()

    baseline_pairs.append((first, second))

    comparison_rows.append({
        "prompt_index": index,
        "prompt": repr(prompt),
        "first_sample_count": int(first.sum()),
        "second_sample_count": int(second.sum()),
        "tv_distance": float(tv),
    })

baseline_comparison = pd.DataFrame(comparison_rows)
observed_mean_tv = baseline_comparison["tv_distance"].mean()

display(baseline_comparison)
print("Mean TV between baseline runs:", observed_mean_tv)

baseline_comparison.to_csv(
    BASELINE_CHECK_DIR / "tv_between_runs.csv",
    index=False,
)

### Original cell 35
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from tqdm.auto import tqdm

rng = np.random.default_rng(2026)
N_PERMUTATIONS = 5000

pooled_samples = []

for first, second in baseline_pairs:
    pooled_counts = first + second

    pool = np.repeat(
        np.arange(len(pooled_counts)),
        pooled_counts,
    )

    pooled_samples.append({
        "pool": pool,
        "pooled_counts": pooled_counts,
        "n_first": int(first.sum()),
        "n_second": int(second.sum()),
    })

null_mean_tv = np.empty(N_PERMUTATIONS)

for iteration in tqdm(
    range(N_PERMUTATIONS),
    desc="Baseline permutation check",
):
    distances = []

    for item in pooled_samples:
        indices = rng.choice(
            len(item["pool"]),
            size=item["n_first"],
            replace=False,
        )

        first_permuted = np.bincount(
            item["pool"][indices],
            minlength=len(item["pooled_counts"]),
        )
        second_permuted = (
            item["pooled_counts"] - first_permuted
        )

        tv = 0.5 * np.abs(
            first_permuted / item["n_first"]
            - second_permuted / item["n_second"]
        ).sum()

        distances.append(tv)

    null_mean_tv[iteration] = np.mean(distances)

p_value = (
    1 + np.count_nonzero(null_mean_tv >= observed_mean_tv)
) / (N_PERMUTATIONS + 1)

report = {
    "observed_mean_tv": float(observed_mean_tv),
    "null_mean_tv": float(null_mean_tv.mean()),
    "null_95th_percentile": float(
        np.quantile(null_mean_tv, 0.95)
    ),
    "permutation_p_value": float(p_value),
    "n_permutations": N_PERMUTATIONS,
    "assumption": (
        "Outputs are exchangeable between runs within each prompt."
    ),
}

(BASELINE_CHECK_DIR / "permutation_report.json").write_text(
    json.dumps(report, indent=2)
)
np.save(
    BASELINE_CHECK_DIR / "permutation_null.npy",
    null_mean_tv,
)

display(pd.DataFrame([report]).drop(columns="assumption"))

### Original cell 36
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

fig_check, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(
    null_mean_tv,
    bins=40,
    color="#8DB9DE",
    edgecolor="white",
    label="Shared-distribution permutation results",
)

ax.axvline(
    observed_mean_tv,
    color="#B53324",
    linewidth=2,
    label=f"Observed mean TV = {observed_mean_tv:.4f}",
)

ax.set_xlabel("Mean TV across the 10 prompts")
ax.set_ylabel("Permutation count")
ax.set_title(
    "Unchanged Qwen2.5-0.5B — two separate collections"
)
ax.legend()
fig_check.tight_layout()

fig_check.savefig(
    BASELINE_CHECK_DIR / "baseline_check.png",
    dpi=300,
)
fig_check.savefig(
    BASELINE_CHECK_DIR / "baseline_check.pdf",
)

plt.show()
print("Permutation p-value:", p_value)
print("Saved in:", BASELINE_CHECK_DIR)

### Original cell 37
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
corrected_difficulty = sweep_results[
    (sweep_results["n_prompts"] == 5)
    & (sweep_results["detection_samples_per_prompt"] == 3)
].copy()

corrected_difficulty.to_csv(
    SWEEP_DIR / "corrected_auc_by_strength.csv",
    index=False,
)

display(corrected_difficulty[["kind", "scale", "auc"]])

### Original cell 38
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
difficulty_results = corrected_difficulty.copy()
ANALYSIS_DIR = SWEEP_DIR

### Original cell 39
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import hashlib
import shutil
import time

LARGE_DIR = NEW_ROOT / "expanded_scales_1000samples_v1"
LARGE_DIR.mkdir(parents=True, exist_ok=True)

LARGE_CONFIG = dict(EXPANDED_CONFIG)
LARGE_CONFIG["n_samples"] = 1000

LARGE_JOBS = [dict(job) for job in EXPANDED_JOBS]
LARGE_CONFIG_PATH = LARGE_DIR / "config.json"

source_worker = EXPANDED_DIR / "worker.py"
LARGE_WORKER = LARGE_DIR / "worker.py"

assert source_worker.exists()

large_manifest = {
    "config": LARGE_CONFIG,
    "jobs": LARGE_JOBS,
    "worker_sha256": hashlib.sha256(
        source_worker.read_bytes()
    ).hexdigest(),
}

manifest_path = LARGE_DIR / "manifest.json"

if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == large_manifest, (
        "Existing experiment has different settings."
    )
else:
    manifest_path.write_text(
        json.dumps(large_manifest, indent=2)
    )

if not LARGE_WORKER.exists():
    shutil.copy2(source_worker, LARGE_WORKER)

assert hashlib.sha256(
    LARGE_WORKER.read_bytes()
).hexdigest() == large_manifest["worker_sha256"]

LARGE_CONFIG_PATH.write_text(
    json.dumps(LARGE_CONFIG, indent=2)
)

print("Model:", LARGE_CONFIG["model"])
print("Prompts:", len(LARGE_CONFIG["prompts"]))
print("Samples per prompt:", LARGE_CONFIG["n_samples"])
print("Model versions:", len(LARGE_JOBS))
print("Results:", LARGE_DIR)

### Original cell 40
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
def collect_large_job(job):
    global WORKER

    output = LARGE_DIR / f"{job['name']}.json"

    if valid_result(output, LARGE_CONFIG, job):
        print("Already complete:", job["name"])
        return None

    job_path = LARGE_DIR / f"{job['name']}_job.json"
    job_path.write_text(json.dumps(job, indent=2))

    checkpoint = WORK / "temporary_1000sample_variant"
    if checkpoint.exists():
        shutil.rmtree(checkpoint)

    arguments = [
        "--config", str(LARGE_CONFIG_PATH),
        "--job", str(job_path),
        "--checkpoint", str(checkpoint),
        "--output", str(output),
    ]

    previous_worker = WORKER
    WORKER = LARGE_WORKER
    started = time.monotonic()

    try:
        if job["kind"] != "unchanged":
            run_logged(
                ["edit", *arguments],
                LARGE_DIR / f"{job['name']}_edit.log",
            )

        run_logged(
            ["sample", *arguments],
            LARGE_DIR / f"{job['name']}_sample.log",
        )

        assert valid_result(output, LARGE_CONFIG, job)

        if checkpoint.exists():
            shutil.rmtree(checkpoint)

    finally:
        WORKER = previous_worker

    elapsed = time.monotonic() - started

    (LARGE_DIR / f"{job['name']}_timing.json").write_text(
        json.dumps({"elapsed_seconds": elapsed}, indent=2)
    )

    print(
        "Completed:", job["name"],
        "| minutes:", round(elapsed / 60, 1),
    )
    return elapsed

### Original cell 41
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
baseline_job = next(
    job for job in LARGE_JOBS
    if job["kind"] == "unchanged"
)

baseline_seconds = collect_large_job(baseline_job)

timing_path = LARGE_DIR / "unchanged_timing.json"

if timing_path.exists():
    baseline_seconds = json.loads(
        timing_path.read_text()
    )["elapsed_seconds"]

    print(
        "\nBaseline collection took approximately",
        round(baseline_seconds / 60, 1),
        "minutes.",
    )
    print(
        "Rough sampling-time estimate for 18 versions:",
        round(18 * baseline_seconds / 3600, 2),
        "hours.",
    )
    print(
        "This is only a rough estimate; variant editing, "
        "loading and inference speed can differ."
    )

### Original cell 42
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
for position, job in enumerate(LARGE_JOBS, start=1):
    print(
        f"\n[{position}/{len(LARGE_JOBS)}]",
        job["name"],
    )
    collect_large_job(job)

print("\nAll collections completed.")

### Original cell 43
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
verification_rows = []

for job in LARGE_JOBS:
    path = LARGE_DIR / f"{job['name']}.json"
    complete = valid_result(path, LARGE_CONFIG, job)

    verification_rows.append({
        "variant": job["name"],
        "complete": complete,
    })

verification_1000 = pd.DataFrame(verification_rows)
display(verification_1000)

assert verification_1000["complete"].all(), (
    "Some collections are incomplete. Rerun Cell 39."
)

verification_1000.to_csv(
    LARGE_DIR / "collection_verification.csv",
    index=False,
)

print("Verified: 18 versions × 10 prompts × 1,000 outputs.")

### Original cell 44
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

LARGE_ANALYSIS_DIR = (
    LARGE_DIR / "figure1_independent_bootstrap_v1"
)
LARGE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

large_collections = {}

for job in LARGE_JOBS:
    path = LARGE_DIR / f"{job['name']}.json"
    assert valid_result(path, LARGE_CONFIG, job), (
        f"Incomplete collection: {job['name']}"
    )
    large_collections[job["name"]] = json.loads(path.read_text())

LARGE_SETTINGS = {
    "n_tests": 1000,
    "reference_samples": 50,
    "prompt_counts": [1, 3, 5, 10],
    "detection_counts": [1, 3, 10, 20, 50],
    "seed": 0,
    "sampling": "independent empirical multinomial draws",
}

analysis_manifest = {
    "settings": LARGE_SETTINGS,
    "inputs": {
        job["name"]: hashlib.sha256(
            (LARGE_DIR / f"{job['name']}.json").read_bytes()
        ).hexdigest()
        for job in LARGE_JOBS
    },
}

path = LARGE_ANALYSIS_DIR / "analysis_manifest.json"
if path.exists():
    assert json.loads(path.read_text()) == analysis_manifest
else:
    path.write_text(json.dumps(analysis_manifest, indent=2))

print("Loaded:", len(large_collections), "model versions")

### Original cell 45
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
def large_auc(baseline, variant, n_prompts, n_detection, seed):
    rng = np.random.default_rng(seed)
    prompts = LARGE_CONFIG["prompts"]
    n_reference = LARGE_SETTINGS["reference_samples"]

    base_probabilities = {}
    variant_probabilities = {}

    for prompt in prompts:
        a_counts = baseline["counts"][prompt]
        b_counts = variant["counts"][prompt]
        tokens = sorted(set(a_counts) | set(b_counts))

        a = np.array(
            [a_counts.get(token, 0) for token in tokens],
            dtype=float,
        )
        b = np.array(
            [b_counts.get(token, 0) for token in tokens],
            dtype=float,
        )

        base_probabilities[prompt] = a / a.sum()
        variant_probabilities[prompt] = b / b.sum()

    def score(selected, left, right):
        distances = []

        for prompt in selected:
            reference = rng.multinomial(
                n_reference, left[prompt]
            ) / n_reference

            detection = rng.multinomial(
                n_detection, right[prompt]
            ) / n_detection

            distances.append(
                0.5 * np.abs(reference - detection).sum()
            )

        return float(np.mean(distances))

    h0, h1 = [], []

    for _ in range(LARGE_SETTINGS["n_tests"]):
        selected = rng.choice(
            prompts, n_prompts, replace=False
        )

        h0.append(score(
            selected, base_probabilities, base_probabilities
        ))
        h0.append(score(
            selected, variant_probabilities, variant_probabilities
        ))
        h1.append(score(
            selected, base_probabilities, variant_probabilities
        ))

    return float(roc_auc_score(
        [0] * len(h0) + [1] * len(h1),
        h0 + h1,
    ))

print("Corrected evaluation ready.")

### Original cell 46
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
large_baseline = large_collections["unchanged"]

control_rows = []

for n_detection in [3, 20, 50]:
    control_rows.append({
        "detection_samples_per_prompt": n_detection,
        "unchanged_vs_unchanged_auc": large_auc(
            large_baseline,
            large_baseline,
            n_prompts=10,
            n_detection=n_detection,
            seed=12345 + n_detection,
        ),
    })

large_controls = pd.DataFrame(control_rows)
display(large_controls)

large_controls.to_csv(
    LARGE_ANALYSIS_DIR / "unchanged_control.csv",
    index=False,
)

assert large_controls[
    "unchanged_vs_unchanged_auc"
].between(0.45, 0.55).all(), (
    "Control is far from 0.5. Stop and share this output."
)

### Original cell 47
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
tasks = [
    (job, n_prompts, n_detection)
    for job in LARGE_JOBS
    if job["kind"] != "unchanged"
    for n_prompts in LARGE_SETTINGS["prompt_counts"]
    for n_detection in LARGE_SETTINGS["detection_counts"]
]

large_rows = []

for job, n_prompts, n_detection in tqdm(
    tasks, desc="1,000-output analysis"
):
    path = LARGE_ANALYSIS_DIR / (
        f"{job['name']}_p{n_prompts}_d{n_detection}.json"
    )

    if path.exists():
        row = json.loads(path.read_text())
    else:
        key = (
            f"large-v1:{LARGE_SETTINGS['seed']}:"
            f"{job['name']}:{n_prompts}:{n_detection}"
        )
        seed = int(
            hashlib.sha256(key.encode()).hexdigest()[:8], 16
        )

        row = {
            "variant": job["name"],
            "kind": job["kind"],
            "scale": job["scale"],
            "n_prompts": n_prompts,
            "detection_samples_per_prompt": n_detection,
            "auc": large_auc(
                large_baseline,
                large_collections[job["name"]],
                n_prompts,
                n_detection,
                seed,
            ),
        }

        temporary = path.with_suffix(".tmp")
        temporary.write_text(json.dumps(row, indent=2))
        temporary.replace(path)

    large_rows.append(row)

large_sweep = pd.DataFrame(large_rows)
assert len(large_sweep) == 340

large_sweep.to_csv(
    LARGE_ANALYSIS_DIR / "auc_per_variant_and_configuration.csv",
    index=False,
)

print("All 340 evaluations completed.")

### Original cell 48
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import matplotlib.pyplot as plt

large_summary = (
    large_sweep
    .groupby(
        ["n_prompts", "detection_samples_per_prompt"],
        as_index=False,
    )
    .agg(
        auc=("auc", "mean"),
        n_variants=("variant", "nunique"),
    )
)

assert len(large_summary) == 20
assert (large_summary["n_variants"] == 17).all()

mean_input_tokens = np.mean([
    large_baseline["prompt_lengths"][prompt]
    for prompt in LARGE_CONFIG["prompts"]
])

# Historical pricing assumptions from the supplied repository.
large_summary["yearly_cost"] = (
    8760
    * large_summary["n_prompts"]
    * large_summary["detection_samples_per_prompt"]
    * (3e-6 * mean_input_tokens + 12e-6)
)

colors = {
    1: "#E8C75A",
    3: "#FFB43B",
    5: "#F56600",
    10: "#963300",
}
sizes = dict(zip([1, 3, 10, 20, 50], [15, 25, 35, 48, 65]))

fig_large, ax = plt.subplots(figsize=(9, 5.5))

for n_prompts, group in large_summary.groupby("n_prompts"):
    group = group.sort_values("yearly_cost")

    ax.plot(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        label=f"{n_prompts} prompts",
    )
    ax.scatter(
        group["yearly_cost"], group["auc"],
        color=colors[n_prompts],
        s=[
            sizes[n]
            for n in group["detection_samples_per_prompt"]
        ],
    )

ax.set_xscale("log")
ax.set_xlabel("Estimated annual monitoring cost ($)")
ax.set_ylabel("Mean ROC AUC over 17 variants")
ax.set_title(
    "Partial Figure 1 — Qwen2.5-0.5B\n"
    "Noise and pruning; 1,000 recorded outputs per prompt"
)
ax.grid(alpha=0.25)
ax.legend(title="Marker size increases\nwith detection samples")

fig_large.text(
    0.12, 0.015,
    "T4 / FP16; 10 Border Inputs; independent empirical bootstrap; "
    "50 reference samples per test.",
    fontsize=8,
)
fig_large.tight_layout(rect=[0, 0.05, 1, 1])

large_summary.to_csv(
    LARGE_ANALYSIS_DIR / "figure1_summary.csv",
    index=False,
)
fig_large.savefig(
    LARGE_ANALYSIS_DIR / "figure1_1000outputs.png",
    dpi=300,
    bbox_inches="tight",
)
fig_large.savefig(
    LARGE_ANALYSIS_DIR / "figure1_1000outputs.pdf",
    bbox_inches="tight",
)

plt.show()
display(large_summary)

### Original cell 49
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
old_path = (
    EXPANDED_DIR
    / "figure1_sweep_v2_independent"
    / "figure1_partial_summary.csv"
)

old_summary = pd.read_csv(old_path)

collection_comparison = old_summary[
    ["n_prompts", "detection_samples_per_prompt", "auc"]
].merge(
    large_summary[
        ["n_prompts", "detection_samples_per_prompt", "auc"]
    ],
    on=["n_prompts", "detection_samples_per_prompt"],
    suffixes=("_100_outputs", "_1000_outputs"),
    validate="one_to_one",
)

collection_comparison["auc_difference"] = (
    collection_comparison["auc_1000_outputs"]
    - collection_comparison["auc_100_outputs"]
)

collection_comparison.to_csv(
    LARGE_ANALYSIS_DIR / "comparison_100_vs_1000_outputs.csv",
    index=False,
)

display(collection_comparison)

print(
    "Mean absolute AUC difference:",
    round(collection_comparison["auc_difference"].abs().mean(), 4),
)

### Original cell 50
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Original cell 51
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/B3IT_Qwen05_Noise_Pruning"
)

if not ROOT.exists():
    raise FileNotFoundError(
        f"Folder not found: {ROOT}\n"
        "Check that you mounted the correct Google Drive account."
    )

print("COLLECTION FOLDERS")
for folder in sorted(ROOT.iterdir()):
    if folder.is_dir():
        json_files = list(folder.glob("*.json"))
        print(f"{folder.name}: {len(json_files)} JSON files")

print("\nANALYSIS FILES")
for path in sorted(ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in {
        ".py", ".ipynb", ".csv"
    }:
        print(path.relative_to(ROOT))

### Original cell 52
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/B3IT_Qwen05_Noise_Pruning"
)
COLLECTION = ROOT / "expanded_scales_1000samples_v1"

records = []
example_shown = False

for path in sorted(COLLECTION.glob("*.json")):
    try:
        saved = json.loads(path.read_text())
    except Exception as error:
        print("Could not read:", path.name, type(error).__name__)
        continue

    if not isinstance(saved, dict):
        continue

    counts = saved.get("counts")
    if not isinstance(counts, dict) or not counts:
        continue

    totals = []
    invalid = False

    for token_counts in counts.values():
        if not isinstance(token_counts, dict):
            invalid = True
            break

        values = list(token_counts.values())
        if not all(
            isinstance(value, int) and value >= 0
            for value in values
        ):
            invalid = True
            break

        totals.append(sum(values))

    records.append({
        "file": path.name,
        "prompts": len(counts),
        "outputs_per_prompt": (
            sorted(set(totals)) if not invalid else "Invalid structure"
        ),
    })

    if not example_shown:
        first_prompt = next(iter(counts))

        print("EXAMPLE FILE:", path.name)
        print("Top-level keys:", list(saved))
        print("First prompt:", repr(first_prompt))
        print(
            "First prompt output counts:",
            dict(Counter(counts[first_prompt]).most_common(5))
        )
        print()
        example_shown = True

print("FILES CONTAINING OUTPUT COUNTS")
display(pd.DataFrame(records))

if not records:
    print("No files with the expected 'counts' structure were found.")

### Original cell 53
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
ANALYSIS = COLLECTION / "figure1_independent_bootstrap_v1"
summary_path = ANALYSIS / "figure1_summary.csv"

if not summary_path.exists():
    raise FileNotFoundError(summary_path)

summary = pd.read_csv(summary_path)

print("Columns:", summary.columns.tolist())
print("Number of configurations:", len(summary))
display(summary)

### Original cell 54
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
from pathlib import Path

COLLECTION = Path(
    "/content/drive/MyDrive/B3IT_Qwen05_Noise_Pruning/"
    "expanded_scales_1000samples_v1"
)

baseline = json.loads(
    (COLLECTION / "unchanged.json").read_text()
)
baseline_prompts = set(baseline["counts"])

collections = {}

for path in sorted(COLLECTION.glob("*.json")):
    saved = json.loads(path.read_text())

    if not isinstance(saved, dict):
        continue
    if not isinstance(saved.get("counts"), dict):
        continue

    assert set(saved["counts"]) == baseline_prompts, (
        f"Different prompts in {path.name}"
    )

    assert all(
        sum(counts.values()) == 1000
        for counts in saved["counts"].values()
    ), f"Unexpected output count in {path.name}"

    collections[path.stem] = saved

assert len(collections) == 18, (
    f"Expected 18 versions, found {len(collections)}"
)

print("Verified: all 18 versions use the same 10 prompts.")
print("\nBASELINE COLLECTION SETTINGS")

# Avoid displaying credentials if any were stored in the config.
def redact(value):
    if isinstance(value, dict):
        return {
            key: (
                "[REDACTED]"
                if any(word in key.lower() for word in (
                    "token", "password", "secret", "api_key"
                )) and not isinstance(item, (int, float, bool))
                else redact(item)
            )
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [redact(item) for item in value]
    return value

print(json.dumps(
    redact(baseline.get("config", {})),
    indent=2,
    ensure_ascii=False,
))

print("\nEXAMPLE PERTURBATION SETTINGS")
example = collections["noise_2pow_minus10"]
print(json.dumps(
    redact(example.get("job", {})),
    indent=2,
    ensure_ascii=False,
))

print("\nBASELINE OUTPUT COUNTS")
for prompt, counts in baseline["counts"].items():
    print(repr(prompt), "->", counts)

### Original cell 55
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import math
import pandas as pd

# Reuse the collections loaded in Cell 5.
assert "collections" in globals(), "Run Cell 5 first."

N_SELECTED = 3
N_DETECTION = 10
N_REFERENCE = 50
N_TESTS = 1000
EXTENSION_SEED = 2026

baseline = collections["unchanged"]

# Preserve the original configured prompt order.
PROMPTS = baseline["config"]["prompts"]

assert len(PROMPTS) == len(set(PROMPTS)) == 10

# Group versions by perturbation type.
families = {}

for name, saved in collections.items():
    if name == "unchanged":
        continue

    kind = saved["job"]["kind"]
    families.setdefault(kind, []).append(name)

assert len(families) == 3, (
    f"Expected three perturbation types, found: {list(families)}"
)

# Define all splits before calculating selection scores.
SPLITS = {}

for held_out_kind in sorted(families):
    selection_variants = sorted(
        name
        for kind, names in families.items()
        if kind != held_out_kind
        for name in names
    )
    evaluation_variants = sorted(families[held_out_kind])

    assert set(selection_variants).isdisjoint(evaluation_variants)

    SPLITS[held_out_kind] = {
        "selection": selection_variants,
        "evaluation": evaluation_variants,
    }

    print(f"\nHeld-out type: {held_out_kind}")
    print("Variants used for selection:", len(selection_variants))
    print("Variants reserved for evaluation:", len(evaluation_variants))

# Describe baseline balance using Shannon entropy.
# Larger entropy means a more balanced output distribution.
balance_rows = []

for prompt in PROMPTS:
    counts = baseline["counts"][prompt]
    total = sum(counts.values())
    probabilities = [
        count / total
        for count in counts.values()
        if count > 0
    ]

    entropy = -sum(
        probability * math.log2(probability)
        for probability in probabilities
    )

    balance_rows.append({
        "prompt": repr(prompt),
        "entropy_bits": entropy,
        "largest_output_fraction": max(probabilities),
    })

balance_table = (
    pd.DataFrame(balance_rows)
    .sort_values("entropy_bits", ascending=False)
    .reset_index(drop=True)
)

print(
    f"\nDetection budget: {N_SELECTED} prompts × "
    f"{N_DETECTION} samples = "
    f"{N_SELECTED * N_DETECTION} requests per test."
)

display(balance_table)

### Original cell 56
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import numpy as np
import pandas as pd

assert "SPLITS" in globals(), "Run Cell 6 first."

split_rng = np.random.default_rng(EXTENSION_SEED)

SELECTION_COUNTS = {}
EVALUATION_COUNTS = {}

for version in sorted(collections):
    saved = collections[version]

    SELECTION_COUNTS[version] = {}
    EVALUATION_COUNTS[version] = {}

    for prompt in PROMPTS:
        counts = saved["counts"][prompt]
        tokens = sorted(counts)

        frequencies = np.array(
            [counts[token] for token in tokens],
            dtype=np.int64,
        )

        assert frequencies.sum() == 1000

        # Select half of the recorded observations without replacement.
        selected = split_rng.multivariate_hypergeometric(
            frequencies, 500
        )
        remaining = frequencies - selected

        SELECTION_COUNTS[version][prompt] = {
            token: int(count)
            for token, count in zip(tokens, selected)
            if count > 0
        }

        EVALUATION_COUNTS[version][prompt] = {
            token: int(count)
            for token, count in zip(tokens, remaining)
            if count > 0
        }

        assert selected.sum() == remaining.sum() == 500
        assert np.array_equal(
            selected + remaining, frequencies
        )

print("Verified: 18 versions × 10 prompts.")
print("Each prompt: 500 selection + 500 evaluation outputs.")
print("Original files unchanged.")

# Recalculate balance using ONLY the selection pool.
# Do not use the full-data entropy table to select prompts.
balance_scores = {}

for prompt in PROMPTS:
    counts = SELECTION_COUNTS["unchanged"][prompt]
    probabilities = np.array(
        list(counts.values()), dtype=float
    )
    probabilities /= probabilities.sum()

    balance_scores[prompt] = float(
        -(probabilities * np.log2(probabilities)).sum()
    )

# Break ties using the configured prompt order.
BALANCE_RANKING = sorted(
    PROMPTS,
    key=lambda prompt: (
        -balance_scores[prompt],
        PROMPTS.index(prompt),
    ),
)

BALANCE_SELECTED = BALANCE_RANKING[:N_SELECTED]

print("\nEntropy-based selection:")
for prompt in BALANCE_SELECTED:
    print(
        repr(prompt),
        "| entropy:",
        round(balance_scores[prompt], 4),
    )

### Original cell 57
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

assert "SELECTION_COUNTS" in globals(), "Run Cell 7 first."

def estimate_prompt_auc(
    baseline_counts,
    changed_counts,
    rng,
    n_tests=N_TESTS,
    n_reference=N_REFERENCE,
    n_detection=N_DETECTION,
):
    tokens = sorted(set(baseline_counts) | set(changed_counts))

    p_original = np.array(
        [baseline_counts.get(token, 0) for token in tokens],
        dtype=float,
    )
    p_changed = np.array(
        [changed_counts.get(token, 0) for token in tokens],
        dtype=float,
    )

    p_original /= p_original.sum()
    p_changed /= p_changed.sum()

    # No-change scores: reference and detection samples
    # independently drawn from the original distribution.
    reference_null = rng.multinomial(
        n_reference, p_original, size=n_tests
    ) / n_reference

    detection_null = rng.multinomial(
        n_detection, p_original, size=n_tests
    ) / n_detection

    # Change scores: original reference versus changed detection.
    reference_changed = rng.multinomial(
        n_reference, p_original, size=n_tests
    ) / n_reference

    detection_changed = rng.multinomial(
        n_detection, p_changed, size=n_tests
    ) / n_detection

    null_scores = 0.5 * np.abs(
        reference_null - detection_null
    ).sum(axis=1)

    changed_scores = 0.5 * np.abs(
        reference_changed - detection_changed
    ).sum(axis=1)

    labels = np.concatenate([
        np.zeros(n_tests),
        np.ones(n_tests),
    ])

    return roc_auc_score(
        labels,
        np.concatenate([null_scores, changed_scores]),
    )


PERFORMANCE_SELECTED = {}
selection_rankings = {}
selection_details = {}

for fold_index, (held_out_kind, split) in enumerate(SPLITS.items()):
    rng = np.random.default_rng(
        EXTENSION_SEED + 100 + fold_index
    )

    rows = []

    for variant in split["selection"]:
        kind = collections[variant]["job"]["kind"]

        assert kind != held_out_kind

        for prompt in PROMPTS:
            auc = estimate_prompt_auc(
                SELECTION_COUNTS["unchanged"][prompt],
                SELECTION_COUNTS[variant][prompt],
                rng,
            )

            rows.append({
                "prompt": prompt,
                "variant": variant,
                "kind": kind,
                "auc": auc,
            })

    details = pd.DataFrame(rows)

    # Average within each type first, then across types.
    # This gives each selection perturbation type equal weight.
    by_type = (
        details.groupby(["prompt", "kind"], as_index=False)["auc"]
        .mean()
    )

    ranking = (
        by_type.groupby("prompt", as_index=False)["auc"]
        .mean()
        .rename(columns={"auc": "selection_auc"})
    )

    prompt_order = {
        prompt: index for index, prompt in enumerate(PROMPTS)
    }
    ranking["prompt_order"] = ranking["prompt"].map(prompt_order)

    ranking = (
        ranking.sort_values(
            ["selection_auc", "prompt_order"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    selected = ranking.head(N_SELECTED)["prompt"].tolist()

    PERFORMANCE_SELECTED[held_out_kind] = selected
    selection_rankings[held_out_kind] = ranking
    selection_details[held_out_kind] = details

    print(f"\nHELD-OUT TYPE: {held_out_kind}")
    print("Selected prompts:", [repr(p) for p in selected])

    shown = ranking[["prompt", "selection_auc"]].copy()
    shown["prompt"] = shown["prompt"].map(repr)
    display(shown)

print("\nSelection completed. Evaluation pool not used.")

### Original cell 58
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

assert "PERFORMANCE_SELECTED" in globals(), "Run Cell 8 first."

ALL_GROUPS = list(combinations(range(len(PROMPTS)), N_SELECTED))
assert len(ALL_GROUPS) == 120

labels = np.concatenate([
    np.zeros(N_TESTS),
    np.ones(N_TESTS),
])

evaluation_rows = []

for fold_index, (held_out_kind, split) in enumerate(SPLITS.items()):
    rng = np.random.default_rng(
        EXTENSION_SEED + 1000 + fold_index
    )

    performance_group = tuple(sorted(
        PROMPTS.index(prompt)
        for prompt in PERFORMANCE_SELECTED[held_out_kind]
    ))

    entropy_group = tuple(sorted(
        PROMPTS.index(prompt)
        for prompt in BALANCE_SELECTED
    ))

    for variant in split["evaluation"]:
        # Only the held-out perturbation type enters evaluation.
        assert collections[variant]["job"]["kind"] == held_out_kind

        null_scores = []
        changed_scores = []

        for prompt in PROMPTS:
            original = EVALUATION_COUNTS["unchanged"][prompt]
            changed = EVALUATION_COUNTS[variant][prompt]

            tokens = sorted(set(original) | set(changed))

            p_original = np.array(
                [original.get(token, 0) for token in tokens],
                dtype=float,
            )
            p_changed = np.array(
                [changed.get(token, 0) for token in tokens],
                dtype=float,
            )

            p_original /= p_original.sum()
            p_changed /= p_changed.sum()

            ref_null = rng.multinomial(
                N_REFERENCE, p_original, size=N_TESTS
            ) / N_REFERENCE

            det_null = rng.multinomial(
                N_DETECTION, p_original, size=N_TESTS
            ) / N_DETECTION

            ref_changed = rng.multinomial(
                N_REFERENCE, p_original, size=N_TESTS
            ) / N_REFERENCE

            det_changed = rng.multinomial(
                N_DETECTION, p_changed, size=N_TESTS
            ) / N_DETECTION

            null_scores.append(
                0.5 * np.abs(ref_null - det_null).sum(axis=1)
            )
            changed_scores.append(
                0.5 * np.abs(ref_changed - det_changed).sum(axis=1)
            )

        null_scores = np.asarray(null_scores)
        changed_scores = np.asarray(changed_scores)

        # Reuse the same simulated prompt scores for every group.
        # This reduces simulation noise in comparisons.
        for group in ALL_GROUPS:
            indices = list(group)

            scores = np.concatenate([
                null_scores[indices].mean(axis=0),
                changed_scores[indices].mean(axis=0),
            ])

            evaluation_rows.append({
                "held_out_type": held_out_kind,
                "variant": variant,
                "group": ",".join(map(str, group)),
                "prompts": repr([PROMPTS[i] for i in group]),
                "auc": roc_auc_score(labels, scores),
                "performance_selected": group == performance_group,
                "entropy_selected": group == entropy_group,
            })

    print("Completed held-out evaluation:", held_out_kind)

evaluation_details = pd.DataFrame(evaluation_rows)

# Average equally over the variants within each held-out type.
group_results = (
    evaluation_details.groupby(
        ["held_out_type", "group", "prompts",
         "performance_selected", "entropy_selected"],
        as_index=False,
    )["auc"]
    .mean()
)

summary_rows = []

for held_out_kind in SPLITS:
    results = group_results[
        group_results["held_out_type"] == held_out_kind
    ]

    performance_auc = results.loc[
        results["performance_selected"], "auc"
    ].item()

    entropy_auc = results.loc[
        results["entropy_selected"], "auc"
    ].item()

    random_mean = results["auc"].mean()

    summary_rows.append({
        "held_out_type": held_out_kind,
        "performance_auc": performance_auc,
        "entropy_auc": entropy_auc,
        "random_mean_auc": random_mean,
        "random_subset_sd": results["auc"].std(ddof=0),
        "gain_vs_entropy": performance_auc - entropy_auc,
        "gain_vs_random": performance_auc - random_mean,
    })

extension_summary = pd.DataFrame(summary_rows)

display(extension_summary.round(4))

# Preserve the results and the exact selected prompts in Drive.
OUTPUT = COLLECTION / "extension_selection_v1"
OUTPUT.mkdir(parents=True, exist_ok=True)

evaluation_details.to_csv(
    OUTPUT / "auc_per_variant_and_subset.csv", index=False
)
group_results.to_csv(
    OUTPUT / "auc_per_subset.csv", index=False
)
extension_summary.to_csv(
    OUTPUT / "strategy_comparison.csv", index=False
)

settings = {
    "seed": EXTENSION_SEED,
    "n_tests": N_TESTS,
    "n_selected": N_SELECTED,
    "n_detection": N_DETECTION,
    "n_reference": N_REFERENCE,
    "outputs_per_pool": 500,
    "prompts": PROMPTS,
    "splits": SPLITS,
    "performance_selected": PERFORMANCE_SELECTED,
    "entropy_selected": BALANCE_SELECTED,
    "balance_score": "Shannon entropy on selection outputs",
}

(OUTPUT / "experiment_settings.json").write_text(
    json.dumps(settings, indent=2, ensure_ascii=False)
)

print("\nSaved:", OUTPUT)

### Original cell 59
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
from pathlib import Path

assert "collections" in globals(), "Run Cell 5 first."
assert "SPLITS" in globals(), "Run Cell 6 first."

# Define these now and keep all five, whatever the results.
REPEAT_SEEDS = [101, 202, 303, 404, 505]

PROTOCOL = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "seeds": REPEAT_SEEDS,
    "candidate_prompts": PROMPTS,
    "selection_pool_size": 500,
    "evaluation_pool_size": 500,
    "selected_prompts": 3,
    "detection_samples_per_prompt": 10,
    "reference_samples_per_prompt": 50,
    "simulation_tests": 1000,
    "detection_requests_per_test": 30,
    "strategies": [
        "performance_based",
        "entropy_based",
        "uniform_random_subset",
    ],
    "selection_aggregation": (
        "Mean prompt AUC within each selection perturbation type, "
        "then equal average across selection types"
    ),
    "evaluation_aggregation": (
        "Mean group AUC across variants within each held-out type"
    ),
    "held_out_splits": SPLITS,
    "primary_metric": "Held-out ROC AUC per perturbation type",
    "comparisons": [
        "Performance AUC minus entropy AUC",
        "Performance AUC minus mean random-subset AUC",
    ],
    "repeat_summary": [
        "Mean and standard deviation across seeds",
        "Minimum and maximum paired AUC differences",
        "Number of seeds with a positive paired difference",
    ],
    "cost_scope": (
        "Equal detection requests only. Discovery, selection and "
        "reference collection costs are excluded."
    ),
    "limitations": [
        "Ten previously collected candidate Border Inputs",
        "Repeated splits of existing outputs, not new inference runs",
        "Entropy baseline not yet verified against repository ranking",
        "Five repetitions do not establish statistical significance",
    ],
}

REPEAT_DIR = COLLECTION / "extension_selection_repeated_v1"
REPEAT_DIR.mkdir(parents=True, exist_ok=True)

protocol_path = REPEAT_DIR / "protocol.json"

# Prevent accidentally changing an existing experiment's protocol.
if protocol_path.exists():
    existing = json.loads(protocol_path.read_text())
    if existing != PROTOCOL:
        raise RuntimeError(
            "A different protocol already exists in this folder. "
            "Use a new experiment folder instead of overwriting it."
        )
    print("Existing matching protocol found.")
else:
    protocol_path.write_text(
        json.dumps(PROTOCOL, indent=2, ensure_ascii=False)
    )
    print("Protocol saved.")

print("\nSeeds:", REPEAT_SEEDS)
print("Selected prompts: 3 out of", len(PROMPTS))
print("Detection requests per test: 30")
print("Reference samples per prompt: 50")
print("Held-out types:", list(SPLITS))
print("\nPrimary metric:", PROTOCOL["primary_metric"])
print("\nSaved:", protocol_path)

### Original cell 60
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import json
import math
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Load the saved protocol.
protocol = json.loads(protocol_path.read_text())

assert protocol["model"] == "Qwen/Qwen2.5-0.5B-Instruct"
assert collections["unchanged"]["config"]["model"] == protocol["model"]

prompts = protocol["candidate_prompts"]
seeds = protocol["seeds"]
splits = protocol["held_out_splits"]

n_selected = protocol["selected_prompts"]
n_detection = protocol["detection_samples_per_prompt"]
n_reference = protocol["reference_samples_per_prompt"]
n_tests = protocol["simulation_tests"]
pool_size = protocol["selection_pool_size"]

groups = list(combinations(range(len(prompts)), n_selected))
group_indices = np.array(groups)

labels = np.concatenate([
    np.zeros(n_tests),
    np.ones(n_tests),
])


def split_recorded_outputs(seed):
    rng = np.random.default_rng(seed)
    selection, evaluation = {}, {}

    for version in sorted(collections):
        selection[version] = {}
        evaluation[version] = {}

        for prompt in prompts:
            counts = collections[version]["counts"][prompt]
            tokens = sorted(counts)

            frequencies = np.array(
                [counts[token] for token in tokens],
                dtype=np.int64,
            )
            assert frequencies.sum() == 2 * pool_size

            first = rng.multivariate_hypergeometric(
                frequencies, pool_size
            )
            second = frequencies - first

            selection[version][prompt] = {
                token: int(count)
                for token, count in zip(tokens, first)
                if count > 0
            }
            evaluation[version][prompt] = {
                token: int(count)
                for token, count in zip(tokens, second)
                if count > 0
            }

    return selection, evaluation


def simulate_scores(pool, variant, rng):
    """Return no-change and change TV scores for each prompt."""
    null_scores, change_scores = [], []

    for prompt in prompts:
        original = pool["unchanged"][prompt]
        changed = pool[variant][prompt]
        tokens = sorted(set(original) | set(changed))

        p0 = np.array(
            [original.get(token, 0) for token in tokens],
            dtype=float,
        )
        p1 = np.array(
            [changed.get(token, 0) for token in tokens],
            dtype=float,
        )
        p0 /= p0.sum()
        p1 /= p1.sum()

        ref0 = rng.multinomial(
            n_reference, p0, size=n_tests
        ) / n_reference
        det0 = rng.multinomial(
            n_detection, p0, size=n_tests
        ) / n_detection

        ref1 = rng.multinomial(
            n_reference, p0, size=n_tests
        ) / n_reference
        det1 = rng.multinomial(
            n_detection, p1, size=n_tests
        ) / n_detection

        null_scores.append(
            0.5 * np.abs(ref0 - det0).sum(axis=1)
        )
        change_scores.append(
            0.5 * np.abs(ref1 - det1).sum(axis=1)
        )

    return np.array(null_scores), np.array(change_scores)


def calculate_auc(null, changed):
    return float(roc_auc_score(
        labels, np.concatenate([null, changed])
    ))


for seed in seeds:
    checkpoint = REPEAT_DIR / f"seed_{seed}.json"

    if checkpoint.exists():
        saved = json.loads(checkpoint.read_text())
        assert saved["protocol"] == protocol
        assert saved["seed"] == seed
        print(f"Seed {seed}: already complete, skipping.", flush=True)
        continue

    print(f"\nStarting seed {seed}", flush=True)
    selection, evaluation = split_recorded_outputs(seed)

    # Entropy selection uses ONLY the selection baseline.
    entropies = []
    for prompt in prompts:
        values = np.array(
            list(selection["unchanged"][prompt].values()),
            dtype=float,
        )
        probabilities = values / values.sum()
        entropies.append(float(
            -(probabilities * np.log2(probabilities)).sum()
        ))

    entropy_indices = sorted(
        range(len(prompts)),
        key=lambda i: (-entropies[i], i),
    )[:n_selected]
    entropy_group = tuple(sorted(entropy_indices))

    summary_rows = []
    subset_rows = []
    selections = {}

    for fold_index, (held_out, split) in enumerate(splits.items()):
        # Separate random streams for selection and evaluation.
        train_rng = np.random.default_rng(
            np.random.SeedSequence([seed, fold_index, 1])
        )
        test_rng = np.random.default_rng(
            np.random.SeedSequence([seed, fold_index, 2])
        )

        # Rank individual prompts using permitted selection types.
        aucs_by_type = {}

        for variant in split["selection"]:
            kind = collections[variant]["job"]["kind"]
            assert kind != held_out

            null, changed = simulate_scores(
                selection, variant, train_rng
            )
            prompt_aucs = np.array([
                calculate_auc(null[i], changed[i])
                for i in range(len(prompts))
            ])
            aucs_by_type.setdefault(kind, []).append(prompt_aucs)

        # Equal weight for each selection perturbation type.
        ranking_scores = np.mean([
            np.mean(values, axis=0)
            for values in aucs_by_type.values()
        ], axis=0)

        ranked_indices = sorted(
            range(len(prompts)),
            key=lambda i: (-ranking_scores[i], i),
        )
        performance_group = tuple(sorted(
            ranked_indices[:n_selected]
        ))

        selections[held_out] = {
            "performance_prompts": [
                prompts[i] for i in performance_group
            ],
            "entropy_prompts": [
                prompts[i] for i in entropy_group
            ],
            "ranking_scores": {
                prompt: float(ranking_scores[i])
                for i, prompt in enumerate(prompts)
            },
        }

        variant_group_aucs = []

        for variant in split["evaluation"]:
            assert collections[variant]["job"]["kind"] == held_out

            null, changed = simulate_scores(
                evaluation, variant, test_rng
            )

            # Shared simulated scores make comparisons paired.
            group_null = null[group_indices].mean(axis=1)
            group_changed = changed[group_indices].mean(axis=1)

            group_aucs = np.array([
                calculate_auc(group_null[i], group_changed[i])
                for i in range(len(groups))
            ])
            variant_group_aucs.append(group_aucs)

        # Equal average over held-out variants within this type.
        mean_group_aucs = np.mean(variant_group_aucs, axis=0)

        performance_auc = float(
            mean_group_aucs[groups.index(performance_group)]
        )
        entropy_auc = float(
            mean_group_aucs[groups.index(entropy_group)]
        )
        random_auc = float(mean_group_aucs.mean())

        summary_rows.append({
            "seed": seed,
            "held_out_type": held_out,
            "performance_auc": performance_auc,
            "entropy_auc": entropy_auc,
            "random_mean_auc": random_auc,
            "random_subset_sd": float(mean_group_aucs.std()),
            "gain_vs_entropy": performance_auc - entropy_auc,
            "gain_vs_random": performance_auc - random_auc,
        })

        for group, auc in zip(groups, mean_group_aucs):
            subset_rows.append({
                "seed": seed,
                "held_out_type": held_out,
                "prompt_indices": list(group),
                "prompts": [prompts[i] for i in group],
                "auc": float(auc),
            })

        print(
            f"  {held_out}: performance={performance_auc:.4f}, "
            f"entropy={entropy_auc:.4f}, random={random_auc:.4f}",
            flush=True,
        )

    # Write a completed seed atomically.
    result = {
        "seed": seed,
        "protocol": protocol,
        "selections": selections,
        "summary": summary_rows,
        "subset_results": subset_rows,
    }

    temporary = checkpoint.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(result, indent=2, ensure_ascii=False)
    )
    temporary.replace(checkpoint)

    print(f"Saved seed {seed}.", flush=True)


# Combine all completed seeds.
all_summary_rows = []

for seed in seeds:
    saved = json.loads(
        (REPEAT_DIR / f"seed_{seed}.json").read_text()
    )
    assert saved["protocol"] == protocol
    all_summary_rows.extend(saved["summary"])

repeated_results = pd.DataFrame(all_summary_rows)
repeated_results.to_csv(
    REPEAT_DIR / "all_seeds_comparison.csv",
    index=False,
)

print("\nAll five seeds completed.")
display(repeated_results.round(4))

### Original cell 61
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import pandas as pd

repeated_results = pd.read_csv(
    REPEAT_DIR / "all_seeds_comparison.csv"
)

expected_types = set(PROTOCOL["held_out_splits"])
expected_seeds = set(PROTOCOL["seeds"])

assert len(repeated_results) == 15
assert not repeated_results.duplicated(
    ["seed", "held_out_type"]
).any()

rows = []

for kind, group in repeated_results.groupby("held_out_type"):
    assert set(group["seed"]) == expected_seeds

    rows.append({
        "held_out_type": kind,
        "n_seeds": len(group),

        "performance_auc_mean": group["performance_auc"].mean(),
        "performance_auc_sd": group["performance_auc"].std(ddof=1),

        "entropy_auc_mean": group["entropy_auc"].mean(),
        "entropy_auc_sd": group["entropy_auc"].std(ddof=1),

        "random_mean_auc": group["random_mean_auc"].mean(),
        "random_mean_auc_sd": group["random_mean_auc"].std(ddof=1),

        "gain_vs_entropy_mean": group["gain_vs_entropy"].mean(),
        "gain_vs_entropy_sd": group["gain_vs_entropy"].std(ddof=1),
        "gain_vs_entropy_min": group["gain_vs_entropy"].min(),
        "gain_vs_entropy_max": group["gain_vs_entropy"].max(),
        "positive_vs_entropy": int(
            (group["gain_vs_entropy"] > 0).sum()
        ),

        "gain_vs_random_mean": group["gain_vs_random"].mean(),
        "gain_vs_random_sd": group["gain_vs_random"].std(ddof=1),
        "gain_vs_random_min": group["gain_vs_random"].min(),
        "gain_vs_random_max": group["gain_vs_random"].max(),
        "positive_vs_random": int(
            (group["gain_vs_random"] > 0).sum()
        ),
    })

metrics_summary = pd.DataFrame(rows)

assert set(metrics_summary["held_out_type"]) == expected_types

metrics_summary.to_csv(
    REPEAT_DIR / "metrics_summary_across_seeds.csv",
    index=False,
)

# Compact table for presentation.
presentation_rows = []

for _, row in metrics_summary.iterrows():
    presentation_rows.append({
        "Held-out type": row["held_out_type"],
        "Performance AUC (mean ± SD)": (
            f"{row['performance_auc_mean']:.4f} ± "
            f"{row['performance_auc_sd']:.4f}"
        ),
        "Entropy AUC (mean ± SD)": (
            f"{row['entropy_auc_mean']:.4f} ± "
            f"{row['entropy_auc_sd']:.4f}"
        ),
        "Random AUC (mean ± SD)": (
            f"{row['random_mean_auc']:.4f} ± "
            f"{row['random_mean_auc_sd']:.4f}"
        ),
        "Mean gain vs entropy": row["gain_vs_entropy_mean"],
        "Mean gain vs random": row["gain_vs_random_mean"],
        "Positive vs entropy": (
            f"{int(row['positive_vs_entropy'])}/"
            f"{int(row['n_seeds'])}"
        ),
        "Positive vs random": (
            f"{int(row['positive_vs_random'])}/"
            f"{int(row['n_seeds'])}"
        ),
    })

presentation_summary = pd.DataFrame(presentation_rows)

display(presentation_summary.round(4))

print(
    "\nSD describes variation across five output splits and simulations."
    "\nIt is not a confidence interval."
    "\nRandom AUC is the average over all 120 three-prompt subsets."
    "\nAll strategies use 30 detection requests per test."
)

### Original cell 62
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path
import subprocess

REPO_CHECK = Path("/content/b3it_source_check")

# Commit used earlier in the reproduction.
SOURCE_COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"

if not REPO_CHECK.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/timothee-chauvin/"
            "token-efficient-change-detection-llm-apis.git",
            str(REPO_CHECK),
        ],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(REPO_CHECK), "checkout", "--detach", SOURCE_COMMIT],
    check=True,
)

result = subprocess.run(
    ["git", "-C", str(REPO_CHECK), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
)

print("Source commit:", result.stdout.strip())
print("Source folder:", REPO_CHECK)

### Original cell 63
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path
import re

# Find repository copies directly under /content.
source_roots = sorted(
    Path("/content").glob(
        "*/track-llm-apis/src/track_llm_apis"
    )
)

if not source_roots:
    print(
        "No repository source found in /content.\n"
        "Your Drive results are still available. "
        "We need to restore the source code to inspect its ranking rule."
    )

pattern = re.compile(
    r"entropy|balanc|sort_values|argsort|sorted\(|"
    r"\.sort\(|most_common|reference_samples_per_prompt",
    re.IGNORECASE,
)

for root in source_roots:
    print("\nSOURCE ROOT:", root)

    candidates = [
        root / "sampling" / "bi_phase_1.py",
        root / "sampling" / "generate.py",
        root / "sampling" / "common.py",
    ]

    for path in candidates:
        if not path.exists():
            continue

        lines = path.read_text().splitlines()
        matches = [
            i for i, line in enumerate(lines)
            if pattern.search(line)
        ]

        if not matches:
            continue

        print("\nFILE:", path.relative_to(root))

        # Show nearby lines, without repeating overlapping sections.
        visible = set()
        for index in matches:
            visible.update(
                range(max(0, index - 7), min(len(lines), index + 9))
            )

        previous = -2
        for index in sorted(visible):
            if index != previous + 1:
                print("...")
            print(f"{index + 1:4}: {lines[index]}")
            previous = index

### Original cell 64
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path
import re

SOURCE = Path(
    "/content/b3it_source_check/"
    "track-llm-apis/src/track_llm_apis"
)

pattern = re.compile(
    r"entropy|balanc|most_common|"
    r"max_border_inputs|reference_samples_per_prompt|"
    r"sort_values|argsort",
    re.IGNORECASE,
)

matches_found = 0

for path in sorted(SOURCE.rglob("*.py")):
    lines = path.read_text().splitlines()
    matches = [
        index for index, line in enumerate(lines)
        if pattern.search(line)
    ]

    if not matches:
        continue

    matches_found += len(matches)
    print("\nFILE:", path.relative_to(SOURCE))

    visible = set()
    for index in matches:
        visible.update(
            range(max(0, index - 5), min(len(lines), index + 7))
        )

    previous = -2
    for index in sorted(visible):
        if index != previous + 1:
            print("...")
        print(f"{index + 1:4}: {lines[index]}")
        previous = index

print("\nMatching lines:", matches_found)

### Original cell 65
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import ast
from pathlib import Path

SOURCE = Path(
    "/content/b3it_source_check/"
    "track-llm-apis/src/track_llm_apis"
)

found = False

for path in sorted(SOURCE.rglob("*.py")):
    source = path.read_text()
    lines = source.splitlines()

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(tree):
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == "get_border_inputs"
        ):
            found = True
            print("\nFILE:", path.relative_to(SOURCE))

            for index in range(node.lineno - 1, node.end_lineno):
                print(f"{index + 1:4}: {lines[index]}")

if not found:
    print("No get_border_inputs definition found.")

### Original cell 66
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import ast
from pathlib import Path

path = Path(
    "/content/b3it_source_check/track-llm-apis/"
    "src/track_llm_apis/sampling/bi_phase_1.py"
)

source = path.read_text()
lines = source.splitlines()
tree = ast.parse(source)

for node in tree.body:
    if isinstance(node, ast.ClassDef):
        methods = [
            item for item in node.body
            if isinstance(item, ast.FunctionDef)
            and item.name == "get_border_inputs"
        ]

        if methods:
            print(f"\nCLASS: {node.name}")
            for index in range(node.lineno - 1, node.end_lineno):
                print(f"{index + 1:4}: {lines[index]}")

### Original cell 67
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
import ast
from pathlib import Path

SOURCE = Path(
    "/content/b3it_source_check/"
    "track-llm-apis/src/track_llm_apis"
)

found = False

for path in sorted(SOURCE.rglob("*.py")):
    source = path.read_text()
    lines = source.splitlines()

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(tree):
        if (
            isinstance(node, ast.ClassDef)
            and node.name == "BIPhase1bOutput"
        ):
            found = True
            print("\nFILE:", path.relative_to(SOURCE))

            for index in range(node.lineno - 1, node.end_lineno):
                print(f"{index + 1:4}: {lines[index]}")

if not found:
    print("Class definition not found. Checking references:")

    for path in sorted(SOURCE.rglob("*.py")):
        for number, line in enumerate(
            path.read_text().splitlines(), start=1
        ):
            if "BIPhase1bOutput" in line:
                print(
                    f"{path.relative_to(SOURCE)}:{number}: {line}"
                )

### Original cell 68
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/content/b3it_source_check")
COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"
URL = (
    "https://github.com/timothee-chauvin/"
    "token-efficient-change-detection-llm-apis.git"
)

def run_git(arguments):
    result = subprocess.run(
        ["git", *arguments],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or result.stdout.strip()
        )
    return result.stdout.strip()

if not REPO.exists():
    print("Source folder missing. Restoring it...")
    run_git(["clone", URL, str(REPO)])
    run_git(["-C", str(REPO), "checkout", "--detach", COMMIT])

print("Commit:", run_git(["-C", str(REPO), "rev-parse", "HEAD"]))
print(
    "Local changes:",
    run_git(["-C", str(REPO), "status", "--short"]) or "None",
)

SOURCE = REPO / "track-llm-apis/src/track_llm_apis"
path = SOURCE / "sampling/generate.py"
lines = path.read_text().splitlines()

print("\nBORDER INPUT LOADING")

terms = (
    "get_border_inputs",
    "max_border_inputs",
    "load_model",
    "BIPhase1",
)

visible = set()

for index, line in enumerate(lines):
    if any(term in line for term in terms):
        visible.update(
            range(max(0, index - 10), min(len(lines), index + 12))
        )

previous = -2
for index in sorted(visible):
    if index != previous + 1:
        print("...")
    print(f"{index + 1:4}: {lines[index]}")
    previous = index

if not visible:
    print("No matching references in generate.py.")

### Original cell 69
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path

REPO = Path("/content/b3it_source_check")
phase_file = (
    REPO / "track-llm-apis/src/track_llm_apis/"
    "sampling/bi_phase_1.py"
)

print("PHASE 1 CLASSES AND ALIASES")
lines = phase_file.read_text().splitlines()

for index, line in enumerate(lines):
    if (
        line.startswith("class ")
        or "BIPhase1bOutput" in line
        or "get_border_inputs" in line
    ):
        start = max(0, index - 2)
        end = min(len(lines), index + 8)

        print("\n...")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")

print("\nALL REFERENCES TO BIPhase1bOutput")

references = []

for path in sorted(REPO.rglob("*.py")):
    for number, line in enumerate(
        path.read_text().splitlines(), start=1
    ):
        if "BIPhase1bOutput" in line:
            references.append(
                f"{path.relative_to(REPO)}:{number}: {line.strip()}"
            )

print("\n".join(references) or "No references found.")

### Original cell 70
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPEAT_DIR = Path(
    "/content/drive/MyDrive/B3IT_Qwen05_Noise_Pruning/"
    "expanded_scales_1000samples_v1/"
    "extension_selection_repeated_v1"
)

results = pd.read_csv(
    REPEAT_DIR / "all_seeds_comparison.csv"
)

types = ["magnitude", "random", "random_noise"]
labels = ["Magnitude pruning", "Random pruning", "Gaussian noise"]

strategies = {
    "performance_auc": ("Performance-based", "#1565C0"),
    "entropy_auc": ("Entropy-based", "#EF6C00"),
    "random_mean_auc": ("Uniform random (mean)", "#607D8B"),
}

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(types))

for offset, (column, (label, color)) in zip(
    [-0.18, 0, 0.18], strategies.items()
):
    grouped = (
        results.groupby("held_out_type")[column]
        .agg(["mean", "std"])
        .reindex(types)
    )

    ax.errorbar(
        x + offset,
        grouped["mean"],
        yerr=grouped["std"],
        fmt="o",
        markersize=7,
        capsize=5,
        color=color,
        label=label,
    )

ax.set_xticks(x, labels)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Mean held-out ROC AUC")
ax.set_xlabel("Perturbation type excluded from prompt selection")
ax.set_title(
    "Qwen2.5-0.5B: Border Input selection\n"
    "3 prompts × 10 detection samples = 30 requests per test"
)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(loc="lower right")

fig.text(
    0.02, 0.02,
    "Points: mean across 5 seeds. Error bars: ±1 SD, not confidence intervals.\n"
    "10 candidate BIs; 50 reference samples per prompt. "
    "Random baseline averages all 120 subsets.\n"
    "Existing-output resampling; selection and reference collection costs excluded.",
    fontsize=9,
)

fig.tight_layout(rect=[0, 0.15, 1, 1])

stem = REPEAT_DIR / "selection_comparison_5seeds"

fig.savefig(
    stem.with_suffix(".png"), dpi=300, bbox_inches="tight"
)
fig.savefig(
    stem.with_suffix(".pdf"), bbox_inches="tight"
)

plt.show()
print("Saved:", stem.with_suffix(".png"))
print("Saved:", stem.with_suffix(".pdf"))

### Original cell 71
Source: `B3IT_Qwen05_Noise_Pruning (2).ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")